# Predicting Hue and Saturation from Raw RGBK Sensor Readings

**A beginner's introduction to Keras / TensorFlow, using our own robot calibration data.**

## What problem are we solving?

Our robots sense color using a photodiode-based color sensor. The sensor gives us four raw
numbers per reading: **R, G, B, K** (red, green, blue, and a clear/broadband channel). What
we actually *want* is the color expressed as **hue** (what color it is, as an angle around a
color wheel, 0 to 1) and **saturation** (how vivid/pure the color is, 0 to 1).

There is a deterministic colorsys formula to go from RGB to HSV. So why use machine learning
at all? Because our sensor is not an ideal RGB camera: the four channels are uncalibrated,
have different gains, and do not sit at the textbook RGB wavelengths. A neural network can
learn the *actual* mapping from raw sensor counts to hue/saturation directly from calibration
data, warts and all, instead of relying on an idealized formula. It can also learn a mapping
that holds across multiple physical sensors, not just one.

## What this notebook is (and is not)

This is **not** a rigorous machine learning study. There is no k-fold cross-validation and no
statistical significance testing. It is intentionally small and readable so that someone who
has never used Keras/TensorFlow before can read every cell, understand every line, and see
a realistic (if miniature) version of the full pipeline: **raw data from several robots ->
train/eval split -> data augmentation -> a small hyperparameter sweep -> a shallow neural
network -> a prediction -> a plot of how good the prediction is -> a check on a robot the
model has never seen at all.**

All of the raw calibration data is pasted directly into this notebook as text (see Step 1),
so there is nothing to upload and no path to configure. Open this in Google Colab, click
"Run all," and it works.

If you are new to this: read every cell's comments before running it. Then try changing a
number (number of neurons, number of epochs, amount of augmentation noise) and re-run, to
build intuition for what each piece does.

## The pipeline, in plain English

1. **Load raw data from three robots** (celeste, tidal, pacific_blue) and pool it together
   for training and evaluation. Pooling matters: a network trained on only one physical
   sensor can quietly learn that one sensor's quirks instead of the general RGBK -> hue/sat
   relationship. A **fourth robot (redwood)** is held out completely and used only at the very
   end, as a check on a sensor the model has truly never seen.
2. **Split into inputs (X) and targets (Y).** X = [R, G, B, K] (4 raw sensor numbers).
   Y = hue and saturation. We train two *separate* small networks (one for hue, one for
   saturation) rather than one network with two outputs, to keep "one input vector -> one
   number" easy to reason about while starting out.
3. **Normalize.** Neural networks train much better when inputs are scaled to a small,
   consistent range (here, roughly [0, 1]) instead of raw sensor counts that might run into
   the thousands.
4. **Handle hue's wraparound.** Hue is circular: 0.99 and 0.01 are neighbors on the color
   wheel, not far apart. A plain single-number regression has no way to represent that, so
   instead of predicting hue directly we predict `sin(2*pi*hue)` and `cos(2*pi*hue)` and
   convert back afterward. This one change matters more than almost anything else in this
   notebook.
5. **Augment the training data** by adding a little Gaussian noise to the RGBK inputs and
   duplicating rows with their target unchanged. This is a cheap way to make the network less
   sensitive to the small sensor noise that is present in every real reading, using only the
   data we already have.
6. **Run a small hyperparameter sweep.** Instead of guessing one network shape, we try a
   handful of candidate hidden-layer sizes, train each one, and keep whichever does best on
   the evaluation set. This is a miniature version of what a real project would do with a much
   larger search.
7. **Evaluate** the chosen network against held-out data from the *same three* robots, and
   separately against redwood, a robot it has never seen in any form. The gap between those
   two numbers is the honest answer to "does this generalize to a new sensor?"
8. **Visualize.** Plot predicted vs. true values, and compare against the plain colorsys
   formula baseline.


## Step 0: Imports

`tensorflow` (with its `keras` submodule) is the library that gives us neural network
building blocks. `pandas` reads the CSV files. `numpy` handles arrays and math.
`matplotlib` makes plots.


In [ ]:
import colorsys
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Make results repeatable: same augmentation noise / same initial network weights every run.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)


## Step 1: Raw RGBK data from four robots (embedded, no upload needed)

Four robots were calibrated the same way: shown a sequence of known hue/saturation colors
and their raw RGBK sensor readings recorded. Each robot has a `_cal` (calibration /
training) reading and, for three of them, a separate `_ver` (verification / eval) reading
collected in a different session.

The next several cells paste that data directly in as text, one comma-separated row per
line, so there is nothing to upload when running this in Colab. Each row has 6 columns, in
this order:

`hue, saturation, R, G, B, K`

We pool **celeste, tidal, and pacific_blue** together: their `_cal` readings become one
combined training set, their `_ver` readings become one combined evaluation set. **redwood**
is set aside entirely and only used at the end, as a robot the model never saw during
training or model selection, which is the most honest test of generalization.


In [ ]:
CELESTE_CAL = """
0.0208,0.9792,718,237,241,1148
0.0625,0.9792,823,374,326,1486
0.1042,0.9792,882,607,429,1912
0.1458,0.9792,1048,956,566,2600
0.1875,0.9792,925,1009,547,2524
0.2292,0.9792,702,881,504,2121
0.2708,0.9792,563,813,491,1895
0.3125,0.9792,481,733,456,1691
0.3542,0.9792,408,617,402,1439
0.3958,0.9792,432,685,492,1625
0.4375,0.9792,503,844,727,2093
0.4792,0.9792,517,893,908,2330
0.5208,0.9792,527,915,1050,2500
0.5625,0.9792,422,761,945,2128
0.6042,0.9792,258,531,746,1528
0.6458,0.9792,152,386,612,1140
0.6875,0.9792,145,329,528,992
0.7292,0.9792,216,314,452,973
0.7708,0.9792,415,452,600,1455
0.8125,0.9792,705,534,662,1881
0.8542,0.9792,752,472,594,1785
0.8958,0.9792,738,400,508,1609
0.9375,0.9792,638,262,319,1179
0.9792,0.9792,688,237,265,1144
0.0208,0.9375,750,262,265,1227
0.0625,0.9375,842,385,342,1534
0.1042,0.9375,865,604,432,1894
0.1458,0.9375,1012,901,546,2464
0.1875,0.9375,962,1036,563,2606
0.2292,0.9375,688,848,490,2056
0.2708,0.9375,605,867,525,2028
0.3125,0.9375,438,657,420,1531
0.3542,0.9375,396,626,429,1468
0.3958,0.9375,468,766,653,1904
0.4375,0.9375,505,863,880,2259
0.4792,0.9375,499,854,868,2233
0.5208,0.9375,521,906,1035,2471
0.5625,0.9375,431,784,971,2188
0.6042,0.9375,264,545,763,1564
0.6458,0.9375,141,339,529,998
0.6875,0.9375,164,370,587,1112
0.7292,0.9375,248,361,517,1116
0.7708,0.9375,401,432,569,1390
0.8125,0.9375,594,559,699,1839
0.8542,0.9375,757,553,681,1966
0.8958,0.9375,711,391,493,1560
0.9375,0.9375,715,306,372,1350
0.9792,0.9375,710,256,288,1207
0.0208,0.8958,736,281,279,1254
0.0625,0.8958,863,455,382,1672
0.1042,0.8958,939,689,492,2119
0.1458,0.8958,1037,963,585,2613
0.1875,0.8958,969,1053,578,2644
0.2292,0.8958,749,917,528,2227
0.2708,0.8958,589,834,508,1959
0.3125,0.8958,534,819,514,1892
0.3542,0.8958,416,643,428,1501
0.3958,0.8958,430,683,514,1641
0.4375,0.8958,492,812,711,2030
0.4792,0.8958,492,836,858,2197
0.5208,0.8958,478,807,921,2211
0.5625,0.8958,414,721,878,2014
0.6042,0.8958,299,561,753,1607
0.6458,0.8958,198,460,701,1350
0.6875,0.8958,175,378,588,1132
0.7292,0.8958,289,419,591,1290
0.7708,0.8958,453,502,655,1600
0.8125,0.8958,568,524,651,1730
0.8542,0.8958,705,547,669,1900
0.8958,0.8958,696,389,487,1538
0.9375,0.8958,714,321,387,1382
0.9792,0.8958,661,249,281,1149
0.0208,0.8542,698,265,263,1184
0.0625,0.8542,740,387,328,1429
0.1042,0.8542,835,605,433,1870
0.1458,0.8542,883,805,492,2201
0.1875,0.8542,921,995,551,2510
0.2292,0.8542,708,850,495,2084
0.2708,0.8542,563,784,481,1854
0.3125,0.8542,525,799,507,1857
0.3542,0.8542,456,698,460,1631
0.3958,0.8542,461,737,559,1773
0.4375,0.8542,491,805,707,2017
0.4792,0.8542,497,843,869,2206
0.5208,0.8542,469,781,889,2141
0.5625,0.8542,409,699,845,1951
0.6042,0.8542,278,488,643,1401
0.6458,0.8542,180,373,550,1094
0.6875,0.8542,150,290,437,868
0.7292,0.8542,306,424,585,1306
0.7708,0.8542,398,424,547,1356
0.8125,0.8542,521,480,592,1579
0.8542,0.8542,649,475,578,1681
0.8958,0.8542,807,475,593,1841
0.9375,0.8542,774,363,435,1529
0.9792,0.8542,661,255,286,1161
0.0208,0.8125,664,269,266,1162
0.0625,0.8125,777,435,364,1552
0.1042,0.8125,846,636,449,1930
0.1458,0.8125,875,802,500,2196
0.1875,0.8125,906,979,556,2480
0.2292,0.8125,689,822,489,2027
0.2708,0.8125,596,819,518,1961
0.3125,0.8125,547,835,543,1954
0.3542,0.8125,474,735,507,1734
0.3958,0.8125,443,696,540,1692
0.4375,0.8125,455,727,645,1838
0.4792,0.8125,478,791,813,2090
0.5208,0.8125,462,764,866,2093
0.5625,0.8125,424,711,848,1984
0.6042,0.8125,323,558,723,1597
0.6458,0.8125,201,394,564,1150
0.6875,0.8125,182,332,488,992
0.7292,0.8125,297,399,539,1225
0.7708,0.8125,445,474,606,1513
0.8125,0.8125,571,532,650,1739
0.8542,0.8125,683,517,623,1803
0.8958,0.8125,698,420,517,1606
0.9375,0.8125,837,424,505,1726
0.9792,0.8125,743,321,356,1377
0.0208,0.7708,748,331,328,1369
0.0625,0.7708,776,455,386,1595
0.1042,0.7708,812,626,453,1891
0.1458,0.7708,921,900,573,2427
0.1875,0.7708,878,973,572,2466
0.2292,0.7708,761,921,554,2269
0.2708,0.7708,538,731,467,1760
0.3125,0.7708,573,868,571,2040
0.3542,0.7708,448,706,493,1666
0.3958,0.7708,417,650,496,1580
0.4375,0.7708,448,706,600,1767
0.4792,0.7708,536,898,848,2303
0.5208,0.7708,517,875,984,2385
0.5625,0.7708,485,845,1007,2341
0.6042,0.7708,347,611,789,1743
0.6458,0.7708,216,434,626,1267
0.6875,0.7708,185,353,521,1049
0.7292,0.7708,199,374,552,1116
0.7708,0.7708,322,428,577,1318
0.8125,0.7708,467,498,635,1588
0.8542,0.7708,595,556,682,1819
0.8958,0.7708,723,534,647,1878
0.9375,0.7708,770,460,566,1763
0.9792,0.7708,869,443,529,1798
0.0208,0.7292,789,380,371,1505
0.0625,0.7292,827,524,442,1776
0.1042,0.7292,919,733,533,2189
0.1458,0.7292,1062,1033,665,2793
0.1875,0.7292,1070,1190,697,3012
0.2292,0.7292,835,1026,623,2526
0.2708,0.7292,656,898,575,2160
0.3125,0.7292,597,920,605,2155
0.3542,0.7292,483,756,529,1788
0.3958,0.7292,458,710,564,1747
0.4375,0.7292,488,784,710,1995
0.4792,0.7292,561,949,967,2493
0.5208,0.7292,551,928,1048,2535
0.5625,0.7292,463,786,936,2187
0.6042,0.7292,365,634,808,1803
0.6458,0.7292,254,472,657,1375
0.6875,0.7292,240,421,601,1254
0.7292,0.7292,354,470,626,1441
0.7708,0.7292,476,513,646,1625
0.8125,0.7292,573,532,644,1737
0.8542,0.7292,694,532,635,1842
0.8958,0.7292,668,415,504,1559
0.9375,0.7292,664,346,404,1382
0.9792,0.7292,719,332,363,1378
0.0208,0.6875,804,415,401,1588
0.0625,0.6875,854,568,482,1889
0.1042,0.6875,938,773,570,2288
0.1458,0.6875,992,978,650,2651
0.1875,0.6875,1036,1158,719,2964
0.2292,0.6875,817,996,638,2489
0.2708,0.6875,676,921,613,2243
0.3125,0.6875,605,909,606,2151
0.3542,0.6875,507,792,565,1885
0.3958,0.6875,491,771,614,1893
0.4375,0.6875,540,873,791,2222
0.4792,0.6875,562,936,956,2467
0.5208,0.6875,530,876,981,2395
0.5625,0.6875,465,767,901,2136
0.6042,0.6875,375,624,781,1777
0.6458,0.6875,298,520,696,1507
0.6875,0.6875,273,440,602,1308
0.7292,0.6875,376,493,642,1502
0.7708,0.6875,507,550,681,1728
0.8125,0.6875,602,567,675,1833
0.8542,0.6875,770,613,722,2087
0.8958,0.6875,746,504,593,1818
0.9375,0.6875,738,409,469,1585
0.9792,0.6875,767,390,421,1544
0.0208,0.6458,873,499,477,1821
0.0625,0.6458,959,686,578,2215
0.1042,0.6458,961,824,617,2410
0.1458,0.6458,1085,1103,753,2979
0.1875,0.6458,1084,1229,788,3157
0.2292,0.6458,818,988,656,2498
0.2708,0.6458,484,623,428,1552
0.3125,0.6458,584,879,607,2100
0.3542,0.6458,528,830,613,1997
0.3958,0.6458,490,757,618,1886
0.4375,0.6458,525,827,762,2130
0.4792,0.6458,538,862,881,2293
0.5208,0.6458,541,871,965,2384
0.5625,0.6458,476,755,869,2102
0.6042,0.6458,397,632,770,1795
0.6458,0.6458,357,589,759,1700
0.6875,0.6458,350,534,701,1578
0.7292,0.6458,438,566,716,1715
0.7708,0.6458,546,600,729,1868
0.8125,0.6458,635,618,725,1975
0.8542,0.6458,706,591,682,1967
0.8958,0.6458,676,486,559,1702
0.9375,0.6458,666,422,468,1534
0.9792,0.6458,821,486,507,1788
0.0208,0.6042,895,552,521,1943
0.0625,0.6042,835,611,518,1953
0.1042,0.6042,938,812,618,2377
0.1458,0.6042,970,969,667,2634
0.1875,0.6042,1071,1205,775,3102
0.2292,0.6042,843,1026,679,2586
0.2708,0.6042,625,819,562,2030
0.3125,0.6042,598,855,603,2084
0.3542,0.6042,587,916,683,2214
0.3958,0.6042,581,908,744,2257
0.4375,0.6042,576,910,844,2349
0.4792,0.6042,576,941,963,2495
0.5208,0.6042,584,959,1058,2614
0.5625,0.6042,523,849,977,2356
0.6042,0.6042,406,651,788,1843
0.6458,0.6042,336,530,674,1535
0.6875,0.6042,319,471,617,1399
0.7292,0.6042,389,492,620,1494
0.7708,0.6042,525,570,690,1777
0.8125,0.6042,660,642,753,2045
0.8542,0.6042,761,635,735,2116
0.8958,0.6042,773,562,643,1957
0.9375,0.6042,771,507,559,1813
0.9792,0.6042,869,517,540,1898
0.0208,0.5625,906,604,566,2057
0.0625,0.5625,828,632,536,1989
0.1042,0.5625,916,816,626,2369
0.1458,0.5625,938,944,662,2570
0.1875,0.5625,1032,1165,769,3013
0.2292,0.5625,915,1105,747,2809
0.2708,0.5625,772,1038,726,2574
0.3125,0.5625,652,942,676,2300
0.3542,0.5625,598,920,699,2243
0.3958,0.5625,623,975,814,2439
0.4375,0.5625,635,1014,941,2613
0.4792,0.5625,632,1035,1056,2741
0.5208,0.5625,648,1054,1155,2872
0.5625,0.5625,544,871,991,2411
0.6042,0.5625,442,699,836,1976
0.6458,0.5625,375,587,737,1695
0.6875,0.5625,369,533,682,1577
0.7292,0.5625,443,564,700,1702
0.7708,0.5625,592,647,776,2009
0.8125,0.5625,686,667,772,2116
0.8542,0.5625,767,650,741,2144
0.8958,0.5625,819,614,694,2107
0.9375,0.5625,853,587,643,2061
0.9792,0.5625,846,537,553,1912
0.0208,0.5208,940,673,623,2223
0.0625,0.5208,954,777,660,2391
0.1042,0.5208,972,900,698,2586
0.1458,0.5208,993,1025,734,2784
0.1875,0.5208,1033,1177,801,3060
0.2292,0.5208,860,1027,711,2634
0.2708,0.5208,794,1049,753,2633
0.3125,0.5208,696,993,723,2446
0.3542,0.5208,535,777,606,1936
0.3958,0.5208,543,800,679,2038
0.4375,0.5208,579,880,822,2297
0.4792,0.5208,622,980,1000,2618
0.5208,0.5208,596,931,1011,2549
0.5625,0.5208,556,864,968,2394
0.6042,0.5208,481,746,876,2103
0.6458,0.5208,413,616,749,1776
0.6875,0.5208,433,646,793,1869
0.7292,0.5208,463,585,710,1753
0.7708,0.5208,591,652,766,2003
0.8125,0.5208,673,657,753,2075
0.8542,0.5208,755,659,742,2144
0.8958,0.5208,779,611,675,2051
0.9375,0.5208,786,564,605,1935
0.9792,0.5208,907,632,643,2162
0.0208,0.4792,1005,763,704,2464
0.0625,0.4792,980,827,703,2514
0.1042,0.4792,1049,1012,792,2875
0.1458,0.4792,1081,1147,833,3102
0.1875,0.4792,1052,1209,838,3148
0.2292,0.4792,933,1123,805,2903
0.2708,0.4792,857,1149,843,2891
0.3125,0.4792,695,979,733,2438
0.3542,0.4792,606,882,698,2208
0.3958,0.4792,615,904,760,2296
0.4375,0.4792,687,1056,976,2741
0.4792,0.4792,709,1123,1115,2972
0.5208,0.4792,603,926,1000,2540
0.5625,0.4792,578,889,984,2460
0.6042,0.4792,520,807,933,2248
0.6458,0.4792,467,700,839,2004
0.6875,0.4792,466,657,796,1917
0.7292,0.4792,507,649,773,1924
0.7708,0.4792,542,629,735,1900
0.8125,0.4792,698,718,818,2227
0.8542,0.4792,793,714,796,2294
0.8958,0.4792,841,724,801,2356
0.9375,0.4792,832,636,679,2132
0.9792,0.4792,941,687,685,2296
0.0208,0.4375,1007,797,734,2533
0.0625,0.4375,966,843,722,2537
0.1042,0.4375,988,969,770,2747
0.1458,0.4375,1038,1095,816,2984
0.1875,0.4375,953,1086,786,2862
0.2292,0.4375,909,1104,815,2868
0.2708,0.4375,744,962,725,2461
0.3125,0.4375,666,929,715,2338
0.3542,0.4375,691,1012,814,2546
0.3958,0.4375,687,1022,880,2618
0.4375,0.4375,680,1016,958,2676
0.4792,0.4375,632,940,947,2533
0.5208,0.4375,693,1081,1152,2943
0.5625,0.4375,623,947,1040,2621
0.6042,0.4375,511,757,861,2132
0.6458,0.4375,461,673,794,1926
0.6875,0.4375,457,617,732,1803
0.7292,0.4375,542,674,782,1995
0.7708,0.4375,618,688,784,2085
0.8125,0.4375,712,715,792,2214
0.8542,0.4375,804,739,811,2346
0.8958,0.4375,851,731,787,2359
0.9375,0.4375,920,743,771,2423
0.9792,0.4375,898,697,688,2271
0.0208,0.3958,1012,856,781,2649
0.0625,0.3958,956,881,754,2601
0.1042,0.3958,945,944,762,2670
0.1458,0.3958,924,975,747,2672
0.1875,0.3958,985,1118,821,2961
0.2292,0.3958,961,1175,877,3056
0.2708,0.3958,776,993,759,2560
0.3125,0.3958,717,971,760,2475
0.3542,0.3958,623,879,722,2246
0.3958,0.3958,606,866,759,2250
0.4375,0.3958,643,940,885,2486
0.4792,0.3958,683,1028,1029,2760
0.5208,0.3958,658,998,1050,2721
0.5625,0.3958,604,889,961,2463
0.6042,0.3958,538,774,861,2177
0.6458,0.3958,496,696,801,1991
0.6875,0.3958,451,591,686,1724
0.7292,0.3958,506,611,699,1813
0.7708,0.3958,595,660,737,1988
0.8125,0.3958,664,674,740,2072
0.8542,0.3958,662,613,663,1930
0.8958,0.3958,752,662,702,2107
0.9375,0.3958,710,580,596,1875
0.9792,0.3958,898,725,708,2322
0.0208,0.3542,1001,887,811,2705
0.0625,0.3542,869,806,702,2383
0.1042,0.3542,939,961,787,2707
0.1458,0.3542,967,1041,817,2855
0.1875,0.3542,976,1123,852,2990
0.2292,0.3542,884,1054,819,2790
0.2708,0.3542,821,1042,823,2718
0.3125,0.3542,680,900,728,2330
0.3542,0.3542,656,905,754,2337
0.3958,0.3542,625,862,764,2268
0.4375,0.3542,664,947,893,2523
0.4792,0.3542,653,948,941,2559
0.5208,0.3542,655,968,1006,2643
0.5625,0.3542,620,888,946,2463
0.6042,0.3542,602,862,942,2414
0.6458,0.3542,546,753,842,2144
0.6875,0.3542,548,728,829,2106
0.7292,0.3542,612,753,842,2207
0.7708,0.3542,671,750,824,2246
0.8125,0.3542,765,791,856,2410
0.8542,0.3542,807,782,832,2417
0.8958,0.3542,832,762,796,2385
0.9375,0.3542,877,773,783,2428
0.9792,0.3542,957,832,809,2595
0.0208,0.3125,1031,965,881,2888
0.0625,0.3125,1018,1001,876,2911
0.1042,0.3125,1073,1147,954,3205
0.1458,0.3125,1077,1194,956,3265
0.1875,0.3125,1038,1218,956,3254
0.2292,0.3125,857,1018,806,2711
0.2708,0.3125,790,987,795,2600
0.3125,0.3125,776,1022,837,2663
0.3542,0.3125,732,992,838,2587
0.3958,0.3125,749,1026,906,2706
0.4375,0.3125,780,1111,1050,2967
0.4792,0.3125,765,1110,1094,2993
0.5208,0.3125,664,960,989,2628
0.5625,0.3125,689,987,1041,2730
0.6042,0.3125,679,956,1035,2681
0.6458,0.3125,662,916,1006,2591
0.6875,0.3125,632,840,935,2412
0.7292,0.3125,628,761,832,2223
0.7708,0.3125,753,855,920,2530
0.8125,0.3125,801,851,907,2561
0.8542,0.3125,831,830,872,2533
0.8958,0.3125,850,806,831,2486
0.9375,0.3125,856,783,786,2422
0.9792,0.3125,944,863,838,2646
0.0208,0.2708,1005,961,883,2860
0.0625,0.2708,1043,1067,939,3069
0.1042,0.2708,997,1070,908,3002
0.1458,0.2708,1108,1250,1019,3419
0.1875,0.2708,1109,1316,1057,3530
0.2292,0.2708,1076,1315,1059,3496
0.2708,0.2708,1000,1275,1041,3360
0.3125,0.2708,908,1191,991,3127
0.3542,0.2708,846,1154,983,3018
0.3958,0.2708,741,1002,901,2667
0.4375,0.2708,771,1080,1017,2893
0.4792,0.2708,704,975,948,2645
0.5208,0.2708,744,1058,1068,2892
0.5625,0.2708,677,932,961,2582
0.6042,0.2708,703,973,1031,2719
0.6458,0.2708,643,865,933,2447
0.6875,0.2708,628,801,868,2301
0.7292,0.2708,722,892,954,2574
0.7708,0.2708,802,918,971,2698
0.8125,0.2708,736,788,827,2353
0.8542,0.2708,736,729,758,2222
0.8958,0.2708,728,690,706,2121
0.9375,0.2708,790,731,728,2247
0.9792,0.2708,867,794,769,2431
0.0208,0.2292,1081,1118,1027,3248
0.0625,0.2292,1086,1152,1026,3291
0.1042,0.2292,1079,1205,1047,3365
0.1458,0.2292,1095,1250,1054,3439
0.1875,0.2292,1057,1260,1052,3412
0.2292,0.2292,963,1164,977,3140
0.2708,0.2292,983,1235,1042,3300
0.3125,0.2292,913,1194,1019,3164
0.3542,0.2292,806,1067,933,2840
0.3958,0.2292,734,965,879,2603
0.4375,0.2292,767,1026,963,2781
0.4792,0.2292,760,1026,990,2799
0.5208,0.2292,729,1000,997,2748
0.5625,0.2292,709,959,976,2661
0.6042,0.2292,715,956,992,2680
0.6458,0.2292,664,863,909,2447
0.6875,0.2292,708,902,957,2578
0.7292,0.2292,752,929,974,2667
0.7708,0.2292,724,834,863,2428
0.8125,0.2292,746,822,846,2423
0.8542,0.2292,787,824,839,2457
0.8958,0.2292,882,913,916,2724
0.9375,0.2292,1002,1032,1003,3056
0.9792,0.2292,899,891,855,2658
0.0208,0.1875,988,1040,958,3004
0.0625,0.1875,978,1058,958,3025
0.1042,0.1875,895,985,868,2769
0.1458,0.1875,977,1102,951,3059
0.1875,0.1875,1027,1211,1034,3309
0.2292,0.1875,940,1134,974,3081
0.2708,0.1875,881,1091,940,2944
0.3125,0.1875,913,1151,1008,3094
0.3542,0.1875,743,946,844,2554
0.3958,0.1875,759,984,903,2667
0.4375,0.1875,719,940,884,2561
0.4792,0.1875,676,889,873,2452
0.5208,0.1875,737,981,967,2703
0.5625,0.1875,724,949,956,2643
0.6042,0.1875,696,898,915,2520
0.6458,0.1875,695,885,917,2506
0.6875,0.1875,706,873,906,2493
0.7292,0.1875,788,944,969,2711
0.7708,0.1875,751,868,884,2512
0.8125,0.1875,796,890,899,2595
0.8542,0.1875,813,879,880,2579
0.8958,0.1875,854,900,888,2650
0.9375,0.1875,882,909,877,2677
0.9792,0.1875,968,997,949,2928
0.0208,0.1458,1020,1129,1038,3211
0.0625,0.1458,1058,1188,1082,3357
0.1042,0.1458,1057,1227,1088,3408
0.1458,0.1458,1040,1213,1066,3355
0.1875,0.1458,930,1089,951,3000
0.2292,0.1458,953,1158,1025,3169
0.2708,0.1458,824,998,891,2736
0.3125,0.1458,762,940,840,2563
0.3542,0.1458,778,979,892,2671
0.3958,0.1458,742,935,861,2557
0.4375,0.1458,840,1082,1018,2965
0.4792,0.1458,744,956,913,2632
0.5208,0.1458,710,907,883,2516
0.5625,0.1458,747,951,942,2656
0.6042,0.1458,690,861,861,2423
0.6458,0.1458,740,920,923,2594
0.6875,0.1458,749,919,931,2611
0.7292,0.1458,742,884,889,2526
0.7708,0.1458,799,923,923,2656
0.8125,0.1458,873,1002,999,2891
0.8542,0.1458,895,999,986,2895
0.8958,0.1458,800,874,852,2537
0.9375,0.1458,836,897,858,2603
0.9792,0.1458,958,1046,980,3005
0.0208,0.1042,1087,1250,1151,3522
0.0625,0.1042,1020,1170,1067,3285
0.1042,0.1042,1043,1219,1113,3408
0.1458,0.1042,1128,1359,1227,3758
0.1875,0.1042,1129,1379,1239,3791
0.2292,0.1042,1055,1296,1166,3556
0.2708,0.1042,1083,1345,1220,3689
0.3125,0.1042,1037,1321,1205,3602
0.3542,0.1042,638,774,718,2142
0.3958,0.1042,822,1024,963,2830
0.4375,0.1042,1014,1322,1254,3629
0.4792,0.1042,1026,1345,1295,3706
0.5208,0.1042,927,1168,1146,3268
0.5625,0.1042,975,1234,1221,3458
0.6042,0.1042,983,1215,1194,3418
0.6458,0.1042,746,883,868,2512
0.6875,0.1042,756,859,842,2468
0.7292,0.1042,656,731,713,2106
0.7708,0.1042,821,945,909,2688
0.8125,0.1042,1069,1281,1210,3594
0.8542,0.1042,1094,1321,1247,3698
0.8958,0.1042,1107,1335,1260,3740
0.9375,0.1042,1005,1177,1115,3323
0.9792,0.1042,1003,1176,1114,3319
0.0208,0.0625,1121,1325,1230,3708
0.0625,0.0625,1097,1292,1200,3622
0.1042,0.0625,906,1043,972,2941
0.1458,0.0625,993,1177,1087,3285
0.1875,0.0625,1070,1286,1176,3568
0.2292,0.0625,1023,1242,1139,3439
0.2708,0.0625,1026,1260,1154,3475
0.3125,0.0625,1055,1322,1221,3637
0.3542,0.0625,997,1244,1150,3425
0.3958,0.0625,1013,1263,1176,3487
0.4375,0.0625,1035,1315,1238,3626
0.4792,0.0625,995,1256,1204,3487
0.5208,0.0625,1008,1278,1228,3548
0.5625,0.0625,972,1220,1181,3400
0.6042,0.0625,951,1183,1157,3314
0.6458,0.0625,991,1231,1202,3452
0.6875,0.0625,1006,1241,1211,3485
0.7292,0.0625,995,1216,1187,3424
0.7708,0.0625,979,1178,1144,3324
0.8125,0.0625,1002,1200,1166,3394
0.8542,0.0625,997,1181,1144,3348
0.8958,0.0625,1028,1220,1172,3449
0.9375,0.0625,1071,1282,1230,3616
0.9792,0.0625,1083,1298,1227,3643
0.0208,0.0208,1147,1397,1314,3900
0.0625,0.0208,1034,1237,1162,3464
0.1042,0.0208,677,786,739,2212
0.1458,0.0208,1067,1318,1237,3659
0.1875,0.0208,966,1154,1082,3228
0.2292,0.0208,701,813,764,2288
0.2708,0.0208,530,601,568,1705
0.3125,0.0208,800,936,882,2633
0.3542,0.0208,1071,1313,1231,3651
0.3958,0.0208,1027,1249,1172,3480
0.4375,0.0208,1018,1242,1176,3468
0.4792,0.0208,884,1058,1003,2967
0.5208,0.0208,760,896,852,2521
0.5625,0.0208,727,856,817,2411
0.6042,0.0208,843,1009,962,2832
0.6458,0.0208,992,1199,1145,3365
0.6875,0.0208,807,951,909,2683
0.7292,0.0208,680,787,753,2230
0.7708,0.0208,714,841,803,2373
0.8125,0.0208,866,1024,978,2886
0.8542,0.0208,557,630,604,1796
0.8958,0.0208,877,1030,974,2900
0.9375,0.0208,1046,1260,1189,3528
0.9792,0.0208,1106,1346,1269,3759
"""

In [ ]:
CELESTE_VER = """
0.05,0.95,820,318,291,1382
0.15,0.95,1039,938,532,2535
0.25,0.95,613,822,470,1933
0.35,0.95,413,624,384,1436
0.45,0.95,449,742,662,1863
0.55,0.95,417,747,920,2082
0.65,0.95,125,331,545,990
0.75,0.95,312,391,553,1246
0.85,0.95,706,514,639,1835
0.95,0.95,788,293,347,1375
0.05,0.85,863,387,344,1553
0.15,0.85,1103,1025,598,2760
0.25,0.85,598,771,445,1840
0.35,0.85,413,640,410,1480
0.45,0.85,448,746,681,1886
0.55,0.85,417,733,892,2041
0.65,0.85,166,376,585,1116
0.75,0.85,338,410,559,1296
0.85,0.85,786,601,742,2106
0.95,0.85,774,317,375,1418
0.05,0.75,871,462,405,1707
0.15,0.75,945,881,541,2391
0.25,0.75,600,775,471,1872
0.35,0.75,470,737,489,1717
0.45,0.75,532,896,823,2268
0.55,0.75,515,910,1090,2521
0.65,0.75,199,397,583,1170
0.75,0.75,370,434,575,1368
0.85,0.75,668,506,609,1762
0.95,0.75,855,402,463,1675
0.05,0.65,921,578,501,1978
0.15,0.65,1056,1041,668,2800
0.25,0.65,707,900,575,2212
0.35,0.65,498,776,544,1838
0.45,0.65,535,872,824,2246
0.55,0.65,484,801,936,2223
0.65,0.65,259,436,593,1280
0.75,0.65,438,513,655,1597
0.85,0.65,741,594,699,2015
0.95,0.65,847,461,512,1783
0.05,0.55,958,688,598,2231
0.15,0.55,1141,1160,786,3127
0.25,0.55,791,987,663,2474
0.35,0.55,542,813,606,1982
0.45,0.55,576,914,876,2382
0.55,0.55,512,812,928,2254
0.65,0.55,329,501,638,1462
0.75,0.55,482,556,679,1709
0.85,0.55,709,595,679,1967
0.95,0.55,921,612,652,2159
0.05,0.45,1078,889,767,2736
0.15,0.45,1159,1223,877,3303
0.25,0.45,907,1149,827,2925
0.35,0.45,654,963,747,2393
0.45,0.45,664,1021,985,2691
0.55,0.45,603,934,1032,2579
0.65,0.45,462,681,822,1963
0.75,0.45,577,681,799,2053
0.85,0.45,786,705,783,2264
0.95,0.45,974,742,765,2465
0.05,0.35,1053,961,841,2865
0.15,0.35,1157,1271,963,3436
0.25,0.35,978,1225,933,3181
0.35,0.35,679,946,775,2424
0.45,0.35,654,942,904,2516
0.55,0.35,656,965,1031,2665
0.65,0.35,574,809,919,2305
0.75,0.35,651,764,852,2267
0.85,0.35,858,835,891,2582
0.95,0.35,997,865,861,2718
0.05,0.25,1104,1115,989,3229
0.15,0.25,1184,1339,1088,3657
0.25,0.25,1053,1316,1062,3479
0.35,0.25,902,1220,1033,3195
0.45,0.25,840,1189,1130,3189
0.55,0.25,814,1162,1200,3199
0.65,0.25,709,964,1042,2727
0.75,0.25,813,983,1045,2850
0.85,0.25,881,915,945,2744
0.95,0.25,1034,997,969,3008
0.05,0.15,1113,1237,1124,3504
0.15,0.15,1185,1383,1192,3804
0.25,0.15,1112,1392,1211,3763
0.35,0.15,866,1116,1005,3015
0.45,0.15,909,1197,1129,3266
0.55,0.15,869,1145,1126,3166
0.65,0.15,843,1082,1106,3051
0.75,0.15,891,1072,1085,3066
0.85,0.15,1043,1188,1181,3435
0.95,0.15,1112,1217,1154,3509
0.05,0.05,1238,1498,1383,4167
0.15,0.05,870,1007,928,2829
0.25,0.05,1202,1503,1379,4134
0.35,0.05,1139,1444,1341,3970
0.45,0.05,1115,1416,1334,3907
0.55,0.05,1053,1314,1257,3658
0.65,0.05,1055,1305,1263,3674
0.75,0.05,1087,1330,1289,3741
0.85,0.05,1017,1198,1150,3394
0.95,0.05,1140,1355,1277,3811
"""

In [ ]:
TIDAL_CAL = """
0.0208,0.9792,638,218,212,1028
0.0625,0.9792,688,317,261,1237
0.1042,0.9792,747,528,352,1624
0.1458,0.9792,853,792,444,2116
0.1875,0.9792,830,933,474,2284
0.2292,0.9792,524,706,392,1651
0.2708,0.9792,487,711,405,1631
0.3125,0.9792,412,643,382,1460
0.3542,0.9792,394,624,379,1418
0.3958,0.9792,399,640,432,1494
0.4375,0.9792,435,725,587,1771
0.4792,0.9792,461,787,758,2031
0.5208,0.9792,473,808,883,2188
0.5625,0.9792,368,646,771,1800
0.6042,0.9792,226,467,644,1343
0.6458,0.9792,127,317,497,941
0.6875,0.9792,137,299,470,908
0.7292,0.9792,225,344,494,1065
0.7708,0.9792,384,424,553,1362
0.8125,0.9792,577,542,660,1778
0.8542,0.9792,585,405,486,1460
0.8958,0.9792,608,326,405,1313
0.9375,0.9792,717,301,356,1334
0.9792,0.9792,634,221,237,1051
0.0208,0.9375,519,184,174,844
0.0625,0.9375,626,296,242,1139
0.1042,0.9375,620,447,297,1363
0.1458,0.9375,769,724,405,1923
0.1875,0.9375,813,927,470,2259
0.2292,0.9375,615,802,431,1885
0.2708,0.9375,440,649,366,1482
0.3125,0.9375,354,541,320,1235
0.3542,0.9375,337,537,330,1222
0.3958,0.9375,330,534,367,1249
0.4375,0.9375,361,602,491,1476
0.4792,0.9375,390,670,645,1726
0.5208,0.9375,381,647,708,1754
0.5625,0.9375,279,483,573,1343
0.6042,0.9375,184,363,492,1043
0.6458,0.9375,105,256,397,758
0.6875,0.9375,104,227,358,688
0.7292,0.9375,184,273,387,845
0.7708,0.9375,310,336,435,1081
0.8125,0.9375,458,434,528,1419
0.8542,0.9375,618,452,542,1601
0.8958,0.9375,504,274,340,1098
0.9375,0.9375,539,231,273,1015
0.9792,0.9375,516,180,194,857
0.0208,0.8958,565,221,209,965
0.0625,0.8958,598,323,257,1160
0.1042,0.8958,753,557,372,1686
0.1458,0.8958,819,767,437,2051
0.1875,0.8958,700,773,404,1916
0.2292,0.8958,533,671,366,1604
0.2708,0.8958,457,657,376,1517
0.3125,0.8958,398,611,361,1393
0.3542,0.8958,325,514,319,1179
0.3958,0.8958,348,560,389,1318
0.4375,0.8958,342,563,465,1389
0.4792,0.8958,360,606,593,1582
0.5208,0.8958,392,661,720,1792
0.5625,0.8958,312,538,634,1494
0.6042,0.8958,205,388,515,1114
0.6458,0.8958,118,264,398,781
0.6875,0.8958,116,244,377,738
0.7292,0.8958,200,282,392,876
0.7708,0.8958,304,328,419,1052
0.8125,0.8958,449,419,507,1375
0.8542,0.8958,616,459,549,1615
0.8958,0.8958,575,323,397,1273
0.9375,0.8958,544,236,279,1030
0.9792,0.8958,526,195,211,901
0.0208,0.8542,569,222,210,969
0.0625,0.8542,637,344,276,1239
0.1042,0.8542,697,522,352,1574
0.1458,0.8542,747,706,406,1883
0.1875,0.8542,786,883,457,2170
0.2292,0.8542,573,711,388,1703
0.2708,0.8542,452,651,374,1505
0.3125,0.8542,400,615,367,1405
0.3542,0.8542,332,523,328,1200
0.3958,0.8542,360,587,416,1384
0.4375,0.8542,391,651,542,1604
0.4792,0.8542,398,678,661,1757
0.5208,0.8542,365,609,664,1653
0.5625,0.8542,309,525,612,1457
0.6042,0.8542,227,416,541,1189
0.6458,0.8542,132,287,424,844
0.6875,0.8542,121,245,371,737
0.7292,0.8542,207,291,398,896
0.7708,0.8542,319,346,438,1102
0.8125,0.8542,440,408,492,1339
0.8542,0.8542,542,403,478,1415
0.8958,0.8542,627,364,444,1412
0.9375,0.8542,587,271,317,1146
0.9792,0.8542,516,202,219,907
0.0208,0.8125,591,251,237,1049
0.0625,0.8125,652,379,298,1314
0.1042,0.8125,704,547,363,1619
0.1458,0.8125,813,786,456,2085
0.1875,0.8125,747,834,441,2062
0.2292,0.8125,577,720,397,1725
0.2708,0.8125,473,675,397,1573
0.3125,0.8125,448,699,420,1595
0.3542,0.8125,359,566,359,1303
0.3958,0.8125,363,582,421,1385
0.4375,0.8125,379,622,521,1542
0.4792,0.8125,392,655,620,1687
0.5208,0.8125,389,651,705,1762
0.5625,0.8125,350,605,702,1672
0.6042,0.8125,284,502,619,1409
0.6458,0.8125,152,313,450,917
0.6875,0.8125,143,281,415,840
0.7292,0.8125,224,311,424,961
0.7708,0.8125,349,377,474,1201
0.8125,0.8125,458,432,518,1408
0.8542,0.8125,524,398,469,1383
0.8958,0.8125,641,380,463,1463
0.9375,0.8125,649,314,366,1298
0.9792,0.8125,576,237,256,1037
0.0208,0.7708,589,275,257,1095
0.0625,0.7708,576,348,280,1191
0.1042,0.7708,574,467,319,1366
0.1458,0.7708,641,620,376,1658
0.1875,0.7708,634,717,397,1782
0.2292,0.7708,527,642,366,1560
0.2708,0.7708,490,684,410,1611
0.3125,0.7708,414,640,392,1470
0.3542,0.7708,367,585,380,1352
0.3958,0.7708,339,533,390,1278
0.4375,0.7708,411,687,586,1709
0.4792,0.7708,412,697,679,1809
0.5208,0.7708,412,692,748,1872
0.5625,0.7708,333,558,638,1541
0.6042,0.7708,259,444,552,1261
0.6458,0.7708,162,314,440,918
0.6875,0.7708,147,266,383,797
0.7292,0.7708,234,314,415,964
0.7708,0.7708,337,358,443,1139
0.8125,0.7708,434,405,480,1317
0.8542,0.7708,528,403,470,1395
0.8958,0.7708,525,318,380,1206
0.9375,0.7708,708,368,424,1471
0.9792,0.7708,608,274,291,1143
0.0208,0.7292,619,309,286,1190
0.0625,0.7292,578,374,300,1242
0.1042,0.7292,667,546,373,1594
0.1458,0.7292,782,773,473,2056
0.1875,0.7292,800,904,499,2249
0.2292,0.7292,638,798,458,1932
0.2708,0.7292,489,677,409,1603
0.3125,0.7292,474,739,458,1704
0.3542,0.7292,362,574,379,1335
0.3958,0.7292,357,565,423,1363
0.4375,0.7292,378,609,520,1527
0.4792,0.7292,440,740,720,1924
0.5208,0.7292,412,683,734,1847
0.5625,0.7292,326,537,610,1483
0.6042,0.7292,268,450,551,1276
0.6458,0.7292,193,350,473,1019
0.6875,0.7292,179,306,426,912
0.7292,0.7292,271,356,462,1092
0.7708,0.7292,359,385,470,1215
0.8125,0.7292,466,436,510,1412
0.8542,0.7292,619,491,568,1672
0.8958,0.7292,637,411,485,1511
0.9375,0.7292,532,285,322,1120
0.9792,0.7292,642,308,323,1246
0.0208,0.6875,638,340,312,1270
0.0625,0.6875,683,467,375,1519
0.1042,0.6875,654,546,382,1591
0.1458,0.6875,736,737,456,1956
0.1875,0.6875,748,841,482,2110
0.2292,0.6875,612,754,445,1844
0.2708,0.6875,495,682,417,1622
0.3125,0.6875,440,659,417,1542
0.3542,0.6875,407,646,435,1512
0.3958,0.6875,373,584,436,1413
0.4375,0.6875,389,619,532,1560
0.4792,0.6875,427,703,683,1834
0.5208,0.6875,396,645,690,1748
0.5625,0.6875,345,561,629,1548
0.6042,0.6875,288,476,575,1347
0.6458,0.6875,212,370,487,1073
0.6875,0.6875,198,324,439,963
0.7292,0.6875,285,366,467,1120
0.7708,0.6875,377,397,478,1252
0.8125,0.6875,471,442,511,1424
0.8542,0.6875,599,481,548,1624
0.8958,0.6875,680,471,543,1684
0.9375,0.6875,522,290,321,1114
0.9792,0.6875,603,309,322,1210
0.0208,0.6458,726,431,391,1532
0.0625,0.6458,756,554,442,1752
0.1042,0.6458,820,734,514,2088
0.1458,0.6458,922,962,606,2534
0.1875,0.6458,796,913,536,2293
0.2292,0.6458,702,870,527,2137
0.2708,0.6458,480,653,414,1573
0.3125,0.6458,465,686,444,1620
0.3542,0.6458,415,658,458,1560
0.3958,0.6458,380,596,458,1459
0.4375,0.6458,362,572,497,1451
0.4792,0.6458,381,614,595,1609
0.5208,0.6458,374,598,634,1622
0.5625,0.6458,349,550,603,1516
0.6042,0.6458,286,458,542,1294
0.6458,0.6458,238,390,495,1127
0.6875,0.6458,243,380,496,1121
0.7292,0.6458,284,367,459,1113
0.7708,0.6458,381,406,482,1269
0.8125,0.6458,459,437,498,1393
0.8542,0.6458,556,449,507,1508
0.8958,0.6458,599,421,475,1483
0.9375,0.6458,541,325,353,1203
0.9792,0.6458,619,343,349,1290
0.0208,0.6042,675,425,381,1466
0.0625,0.6042,697,530,425,1650
0.1042,0.6042,740,654,471,1878
0.1458,0.6042,848,872,561,2316
0.1875,0.6042,914,1065,641,2676
0.2292,0.6042,669,834,520,2061
0.2708,0.6042,492,659,426,1603
0.3125,0.6042,493,728,480,1730
0.3542,0.6042,454,715,504,1702
0.3958,0.6042,449,703,544,1723
0.4375,0.6042,398,623,544,1586
0.4792,0.6042,422,677,658,1779
0.5208,0.6042,432,695,733,1881
0.5625,0.6042,365,572,629,1580
0.6042,0.6042,307,485,564,1366
0.6458,0.6042,254,400,498,1157
0.6875,0.6042,263,384,488,1138
0.7292,0.6042,322,408,498,1232
0.7708,0.6042,418,455,534,1410
0.8125,0.6042,478,464,524,1468
0.8542,0.6042,581,425,469,1466
0.8958,0.6042,567,394,428,1378
0.9375,0.6042,634,379,374,1377
0.9792,0.6042,703,421,420,1526
0.0208,0.5625,771,536,476,1778
0.0625,0.5625,704,562,451,1722
0.1042,0.5625,753,703,510,1987
0.1458,0.5625,810,848,564,2258
0.1875,0.5625,802,928,580,2354
0.2292,0.5625,709,870,554,2173
0.2708,0.5625,553,744,492,1820
0.3125,0.5625,463,658,446,1592
0.3542,0.5625,448,684,490,1647
0.3958,0.5625,464,717,566,1772
0.4375,0.5625,502,801,703,2037
0.4792,0.5625,522,859,832,2246
0.5208,0.5625,483,771,806,2083
0.5625,0.5625,423,670,725,1837
0.6042,0.5625,333,515,591,1449
0.6458,0.5625,283,435,526,1250
0.6875,0.5625,286,409,505,1206
0.7292,0.5625,356,453,541,1355
0.7708,0.5625,446,485,558,1493
0.8125,0.5625,503,487,542,1533
0.8542,0.5625,587,502,552,1640
0.8958,0.5625,592,448,486,1519
0.9375,0.5625,568,390,410,1358
0.9792,0.5625,657,422,416,1483
0.0208,0.5208,731,532,468,1727
0.0625,0.5208,827,694,557,2088
0.1042,0.5208,848,818,598,2293
0.1458,0.5208,839,895,602,2375
0.1875,0.5208,901,1064,680,2702
0.2292,0.5208,776,967,627,2418
0.2708,0.5208,640,867,585,2131
0.3125,0.5208,463,650,451,1588
0.3542,0.5208,424,623,459,1528
0.3958,0.5208,433,645,516,1616
0.4375,0.5208,447,681,600,1750
0.4792,0.5208,466,728,704,1921
0.5208,0.5208,450,696,718,1885
0.5625,0.5208,404,616,658,1693
0.6042,0.5208,358,546,612,1527
0.6458,0.5208,304,446,523,1280
0.6875,0.5208,309,432,517,1264
0.7292,0.5208,368,456,531,1360
0.7708,0.5208,450,494,559,1507
0.8125,0.5208,552,541,595,1692
0.8542,0.5208,566,495,537,1598
0.8958,0.5208,592,473,502,1563
0.9375,0.5208,586,426,437,1441
0.9792,0.5208,672,465,448,1577
0.0208,0.4792,790,614,537,1944
0.0625,0.4792,797,693,558,2061
0.1042,0.4792,784,766,567,2141
0.1458,0.4792,872,944,645,2502
0.1875,0.4792,772,894,586,2293
0.2292,0.4792,706,860,582,2187
0.2708,0.4792,639,849,587,2112
0.3125,0.4792,493,692,490,1703
0.3542,0.4792,495,733,546,1802
0.3958,0.4792,539,819,657,2047
0.4375,0.4792,530,804,713,2075
0.4792,0.4792,517,799,770,2112
0.5208,0.4792,475,724,741,1962
0.5625,0.4792,480,737,778,2016
0.6042,0.4792,411,623,690,1738
0.6458,0.4792,347,513,590,1460
0.6875,0.4792,336,462,540,1336
0.7292,0.4792,411,513,585,1517
0.7708,0.4792,463,509,565,1544
0.8125,0.4792,558,559,602,1726
0.8542,0.4792,589,530,567,1688
0.8958,0.4792,610,502,526,1637
0.9375,0.4792,615,477,481,1571
0.9792,0.4792,745,555,527,1824
0.0208,0.4375,689,556,485,1735
0.0625,0.4375,699,615,501,1827
0.1042,0.4375,763,760,571,2118
0.1458,0.4375,808,865,608,2316
0.1875,0.4375,879,1033,696,2657
0.2292,0.4375,829,1041,715,2639
0.2708,0.4375,725,987,691,2450
0.3125,0.4375,614,887,642,2181
0.3542,0.4375,556,825,623,2038
0.3958,0.4375,505,748,604,1886
0.4375,0.4375,494,740,661,1921
0.4792,0.4375,517,788,756,2088
0.5208,0.4375,477,725,735,1960
0.5625,0.4375,464,690,720,1894
0.6042,0.4375,422,628,683,1747
0.6458,0.4375,367,534,604,1517
0.6875,0.4375,397,546,623,1576
0.7292,0.4375,413,512,571,1505
0.7708,0.4375,508,577,630,1725
0.8125,0.4375,563,577,616,1763
0.8542,0.4375,621,578,609,1813
0.8958,0.4375,664,574,594,1834
0.9375,0.4375,658,530,526,1712
0.9792,0.4375,709,558,526,1792
0.0208,0.3958,734,624,542,1905
0.0625,0.3958,761,704,572,2051
0.1042,0.3958,693,693,534,1940
0.1458,0.3958,740,784,572,2124
0.1875,0.3958,788,915,633,2377
0.2292,0.3958,757,931,657,2386
0.2708,0.3958,614,794,573,2013
0.3125,0.3958,564,768,566,1929
0.3542,0.3958,503,719,557,1805
0.3958,0.3958,520,753,622,1922
0.4375,0.3958,554,821,731,2136
0.4792,0.3958,519,772,733,2050
0.5208,0.3958,496,751,752,2028
0.5625,0.3958,475,698,718,1912
0.6042,0.3958,434,627,666,1744
0.6458,0.3958,389,550,607,1559
0.6875,0.3958,398,533,595,1538
0.7292,0.3958,442,543,596,1590
0.7708,0.3958,491,552,593,1645
0.8125,0.3958,543,555,587,1693
0.8542,0.3958,610,568,595,1777
0.8958,0.3958,675,586,604,1867
0.9375,0.3958,660,533,529,1721
0.9792,0.3958,701,552,519,1772
0.0208,0.3542,763,692,603,2073
0.0625,0.3542,710,680,562,1970
0.1042,0.3542,709,736,573,2044
0.1458,0.3542,740,813,603,2190
0.1875,0.3542,743,868,626,2277
0.2292,0.3542,672,816,599,2125
0.2708,0.3542,600,765,571,1966
0.3125,0.3542,577,781,596,1984
0.3542,0.3542,483,671,527,1705
0.3958,0.3542,491,683,571,1769
0.4375,0.3542,550,798,714,2092
0.4792,0.3542,494,713,669,1901
0.5208,0.3542,475,691,682,1870
0.5625,0.3542,475,680,689,1864
0.6042,0.3542,442,623,648,1729
0.6458,0.3542,428,591,631,1665
0.6875,0.3542,374,481,521,1385
0.7292,0.3542,423,501,532,1464
0.7708,0.3542,510,563,589,1672
0.8125,0.3542,567,579,597,1753
0.8542,0.3542,617,601,603,1831
0.8958,0.3542,674,614,599,1894
0.9375,0.3542,635,559,529,1728
0.9792,0.3542,738,651,602,2000
0.0208,0.3125,761,710,619,2103
0.0625,0.3125,829,819,681,2351
0.1042,0.3125,885,935,750,2605
0.1458,0.3125,927,1056,795,2828
0.1875,0.3125,859,1040,767,2718
0.2292,0.3125,684,841,626,2189
0.2708,0.3125,631,800,608,2072
0.3125,0.3125,557,738,570,1895
0.3542,0.3125,509,697,558,1791
0.3958,0.3125,513,705,589,1833
0.4375,0.3125,510,713,641,1889
0.4792,0.3125,478,679,650,1830
0.5208,0.3125,495,701,701,1920
0.5625,0.3125,497,701,715,1934
0.6042,0.3125,470,654,674,1816
0.6458,0.3125,441,598,625,1679
0.6875,0.3125,442,569,603,1627
0.7292,0.3125,482,586,611,1692
0.7708,0.3125,529,599,614,1754
0.8125,0.3125,559,600,611,1788
0.8542,0.3125,643,643,643,1941
0.8958,0.3125,655,626,614,1905
0.9375,0.3125,612,562,537,1718
0.9792,0.3125,727,665,616,2017
0.0208,0.2708,806,793,690,2314
0.0625,0.2708,783,813,676,2299
0.1042,0.2708,782,847,678,2337
0.1458,0.2708,870,984,760,2654
0.1875,0.2708,936,1125,853,2968
0.2292,0.2708,889,1110,843,2898
0.2708,0.2708,795,1031,790,2668
0.3125,0.2708,712,946,744,2445
0.3542,0.2708,674,923,742,2378
0.3958,0.2708,686,956,811,2494
0.4375,0.2708,601,841,748,2223
0.4792,0.2708,590,826,761,2208
0.5208,0.2708,546,768,736,2076
0.5625,0.2708,529,730,715,1996
0.6042,0.2708,519,711,716,1966
0.6458,0.2708,490,661,679,1848
0.6875,0.2708,504,650,673,1844
0.7292,0.2708,543,672,685,1918
0.7708,0.2708,595,683,689,1984
0.8125,0.2708,596,646,645,1902
0.8542,0.2708,625,645,635,1917
0.8958,0.2708,625,621,600,1858
0.9375,0.2708,664,637,599,1912
0.9792,0.2708,711,668,613,2005
0.0208,0.2292,818,834,727,2402
0.0625,0.2292,841,896,756,2523
0.1042,0.2292,842,941,774,2593
0.1458,0.2292,769,870,695,2368
0.1875,0.2292,822,977,771,2612
0.2292,0.2292,806,984,779,2610
0.2708,0.2292,741,929,740,2449
0.3125,0.2292,644,829,670,2175
0.3542,0.2292,665,877,725,2301
0.3958,0.2292,783,1084,927,2844
0.4375,0.2292,748,1039,921,2754
0.4792,0.2292,739,1041,948,2775
0.5208,0.2292,670,945,891,2544
0.5625,0.2292,644,888,856,2423
0.6042,0.2292,631,859,846,2369
0.6458,0.2292,556,725,727,2030
0.6875,0.2292,626,810,816,2278
0.7292,0.2292,622,763,758,2165
0.7708,0.2292,639,739,727,2126
0.8125,0.2292,702,794,777,2295
0.8542,0.2292,651,699,678,2046
0.8958,0.2292,666,691,661,2034
0.9375,0.2292,680,693,641,2030
0.9792,0.2292,708,708,646,2079
0.0208,0.1875,787,834,730,2378
0.0625,0.1875,972,1107,941,3070
0.1042,0.1875,883,1025,848,2802
0.1458,0.1875,855,1005,816,2721
0.1875,0.1875,875,1062,854,2843
0.2292,0.1875,755,929,753,2478
0.2708,0.1875,706,877,717,2335
0.3125,0.1875,676,848,706,2272
0.3542,0.1875,611,786,663,2089
0.3958,0.1875,644,845,734,2255
0.4375,0.1875,623,815,723,2189
0.4792,0.1875,636,845,768,2278
0.5208,0.1875,686,925,860,2507
0.5625,0.1875,671,897,853,2455
0.6042,0.1875,667,885,857,2441
0.6458,0.1875,521,659,649,1847
0.6875,0.1875,641,813,803,2284
0.7292,0.1875,659,807,787,2278
0.7708,0.1875,720,853,825,2425
0.8125,0.1875,698,796,762,2280
0.8542,0.1875,711,785,747,2265
0.8958,0.1875,701,748,693,2162
0.9375,0.1875,738,782,714,2258
0.9792,0.1875,801,854,768,2450
0.0208,0.1458,,,,
0.0625,0.1458,806,894,779,2511
0.1042,0.1458,849,961,829,2674
0.1458,0.1458,824,953,802,2617
0.1875,0.1458,843,986,825,2694
0.2292,0.1458,856,1030,850,2783
0.2708,0.1458,775,951,796,2561
0.3125,0.1458,740,913,763,2453
0.3542,0.1458,770,981,825,2617
0.3958,0.1458,733,944,811,2526
0.4375,0.1458,709,916,797,2458
0.4792,0.1458,678,876,781,2369
0.5208,0.1458,670,874,788,2365
0.5625,0.1458,704,920,846,2505
0.6042,0.1458,692,897,837,2459
0.6458,0.1458,688,882,835,2436
0.6875,0.1458,658,826,788,2299
0.7292,0.1458,656,815,786,2284
0.7708,0.1458,689,836,801,2353
0.8125,0.1458,734,865,823,2450
0.8542,0.1458,744,867,820,2460
0.8958,0.1458,718,806,762,2311
0.9375,0.1458,760,838,776,2400
0.9792,0.1458,750,822,746,2344
0.0208,0.1042,809,887,788,2514
0.0625,0.1042,1031,1215,1061,3366
0.1042,0.1042,1011,1205,1045,3319
0.1458,0.1042,1021,1237,1062,3381
0.1875,0.1042,1042,1286,1091,3485
0.2292,0.1042,1037,1296,1092,3493
0.2708,0.1042,985,1245,1052,3345
0.3125,0.1042,909,1164,991,3122
0.3542,0.1042,816,1044,896,2804
0.3958,0.1042,803,1035,896,2779
0.4375,0.1042,846,1095,957,2948
0.4792,0.1042,883,1156,1023,3116
0.5208,0.1042,873,1147,1031,3103
0.5625,0.1042,837,1102,1003,2992
0.6042,0.1042,815,1059,976,2897
0.6458,0.1042,756,968,900,2665
0.6875,0.1042,757,959,900,2656
0.7292,0.1042,734,909,852,2532
0.7708,0.1042,755,914,852,2556
0.8125,0.1042,772,924,859,2592
0.8542,0.1042,774,911,843,2565
0.8958,0.1042,922,1089,989,3048
0.9375,0.1042,820,950,853,2662
0.9792,0.1042,967,1132,1003,3151
0.0208,0.0625,980,1148,1018,3197
0.0625,0.0625,1013,1234,1084,3391
0.1042,0.0625,993,1206,1060,3317
0.1458,0.0625,1017,1255,1094,3428
0.1875,0.0625,1030,1282,1109,3487
0.2292,0.0625,985,1242,1073,3363
0.2708,0.0625,997,1264,1100,3424
0.3125,0.0625,965,1245,1084,3356
0.3542,0.0625,929,1190,1039,3215
0.3958,0.0625,940,1205,1059,3264
0.4375,0.0625,930,1199,1061,3248
0.4792,0.0625,949,1233,1096,3339
0.5208,0.0625,941,1215,1099,3314
0.5625,0.0625,904,1167,1056,3182
0.6042,0.0625,916,1173,1071,3214
0.6458,0.0625,950,1202,1099,3308
0.6875,0.0625,933,1169,1068,3225
0.7292,0.0625,941,1163,1058,3217
0.7708,0.0625,987,1210,1097,3350
0.8125,0.0625,944,1145,1031,3172
0.8542,0.0625,972,1188,1059,3275
0.8958,0.0625,976,1189,1054,3276
0.9375,0.0625,991,1210,1072,3332
0.9792,0.0625,991,1211,1073,3333
0.0208,0.0208,991,1211,1073,3334
0.0625,0.0208,1025,1273,1132,3494
0.1042,0.0208,996,1237,1095,3389
0.1458,0.0208,1012,1261,1118,3454
0.1875,0.0208,1015,1291,1147,3535
0.2292,0.0208,1036,1294,1144,3536
0.2708,0.0208,958,1193,1056,3265
0.3125,0.0208,1070,1363,1200,3709
0.3542,0.0208,1010,1269,1121,3464
0.3958,0.0208,1021,1291,1143,3520
0.4375,0.0208,1008,1271,1126,3469
0.4792,0.0208,1023,1296,1158,3544
0.5208,0.0208,1040,1324,1183,3615
0.5625,0.0208,1005,1277,1143,3490
0.6042,0.0208,979,1244,1120,3405
0.6458,0.0208,965,1218,1096,3340
0.6875,0.0208,948,1188,1072,3272
0.7292,0.0208,993,1247,1123,3425
0.7708,0.0208,934,1165,1051,3205
0.8125,0.0208,896,1114,1005,3074
0.8542,0.0208,936,1159,1046,3196
0.8958,0.0208,927,1147,1035,3162
0.9375,0.0208,925,1138,1016,3130
0.9792,0.0208,800,971,869,2681
,,984,1242,1108,3411
"""

In [ ]:
TIDAL_VER = """
0.05,0.95,745,296,258,1264
0.15,0.95,1042,989,523,2595
0.25,0.95,642,903,484,2070
0.35,0.95,442,707,408,1585
0.45,0.95,452,763,641,1880
0.55,0.95,411,736,870,2036
0.65,0.95,118,301,488,907
0.75,0.95,311,388,539,1240
0.85,0.95,679,508,615,1791
0.95,0.95,744,284,325,1306
0.05,0.85,850,397,331,1543
0.15,0.85,982,933,515,2466
0.25,0.85,642,869,470,2019
0.35,0.85,433,705,423,1587
0.45,0.85,468,800,689,1986
0.55,0.85,413,724,845,1998
0.65,0.85,149,326,497,974
0.75,0.85,317,382,509,1208
0.85,0.85,712,547,654,1904
0.95,0.85,734,310,355,1358
0.05,0.75,834,458,379,1648
0.15,0.75,1108,1111,633,2904
0.25,0.75,689,944,535,2213
0.35,0.75,494,809,502,1839
0.45,0.75,532,908,788,2262
0.55,0.75,461,784,897,2163
0.65,0.75,207,410,590,1212
0.75,0.75,358,422,545,1325
0.85,0.75,677,526,614,1810
0.95,0.75,832,401,448,1642
0.05,0.65,887,573,472,1919
0.15,0.65,1077,1113,665,2907
0.25,0.65,800,1087,650,2592
0.35,0.65,541,883,581,2043
0.45,0.65,535,890,793,2250
0.55,0.65,499,835,933,2291
0.65,0.65,255,429,568,1257
0.75,0.65,412,482,598,1494
0.85,0.65,672,548,622,1837
0.95,0.65,803,445,478,1700
0.05,0.55,916,679,559,2153
0.15,0.55,1104,1161,739,3057
0.25,0.55,865,1131,709,2761
0.35,0.55,584,913,637,2172
0.45,0.55,594,965,874,2470
0.55,0.55,546,880,962,2414
0.65,0.55,333,511,629,1480
0.75,0.55,456,526,621,1608
0.85,0.55,694,598,657,1948
0.95,0.55,743,493,506,1727
0.05,0.45,1043,882,720,2660
0.15,0.45,1176,1293,866,3400
0.25,0.45,846,1087,738,2721
0.35,0.45,631,939,686,2295
0.45,0.45,635,978,895,2542
0.55,0.45,548,841,884,2298
0.65,0.45,452,668,776,1911
0.75,0.45,529,625,704,1867
0.85,0.45,743,677,722,2148
0.95,0.45,834,637,630,2096
0.05,0.35,1001,932,773,2729
0.15,0.35,1040,1162,833,3089
0.25,0.35,970,1244,889,3166
0.35,0.35,661,934,723,2354
0.45,0.35,633,925,841,2433
0.55,0.35,630,929,944,2534
0.65,0.35,522,731,796,2066
0.75,0.35,607,713,762,2096
0.85,0.35,747,730,745,2233
0.95,0.35,877,767,727,2375
0.05,0.25,1065,1101,924,3130
0.15,0.25,1180,1366,1046,3661
0.25,0.25,1070,1375,1044,3561
0.35,0.25,883,1217,972,3132
0.45,0.25,814,1162,1047,3075
0.55,0.25,678,948,931,2590
0.65,0.25,606,810,835,2275
0.75,0.25,711,854,866,2455
0.85,0.25,819,859,847,2545
0.95,0.25,905,874,810,2608
0.05,0.15,1134,1293,1112,3596
0.15,0.15,1166,1394,1133,3762
0.25,0.15,1069,1353,1113,3602
0.35,0.15,859,1129,962,2997
0.45,0.15,835,1101,984,2964
0.55,0.15,804,1063,993,2901
0.65,0.15,795,1029,999,2863
0.75,0.15,898,1094,1053,3086
0.85,0.15,1049,1217,1151,3466
0.95,0.15,1098,1229,1104,3480
0.05,0.05,1214,1491,1301,4083
0.15,0.05,1203,1487,1286,4052
0.25,0.05,1184,1500,1299,4061
0.35,0.05,1120,1454,1274,3922
0.45,0.05,1108,1458,1299,3957
0.55,0.05,1094,1397,1264,3821
0.65,0.05,1070,1364,1250,3749
0.75,0.05,1080,1342,1232,3716
0.85,0.05,1095,1324,1205,3684
0.95,0.05,1163,1424,1270,3931
"""

In [ ]:
PACIFIC_BLUE_CAL = """
0.0208,0.9792,656,218,219,1048
0.0625,0.9792,723,329,280,1298
0.1042,0.9792,789,547,377,1699
0.1458,0.9792,887,798,466,2159
0.1875,0.9792,830,899,481,2229
0.2292,0.9792,648,795,441,1864
0.2708,0.9792,505,707,417,1639
0.3125,0.9792,439,655,397,1499
0.3542,0.9792,409,625,391,1431
0.3958,0.9792,411,636,442,1494
0.4375,0.9792,439,692,498,1637
0.4792,0.9792,468,766,737,1976
0.5208,0.9792,375,633,741,1745
0.5625,0.9792,358,616,745,1711
0.6042,0.9792,213,420,581,1206
0.6458,0.9792,127,309,486,914
0.6875,0.9792,137,291,463,885
0.7292,0.9792,229,336,483,1040
0.7708,0.9792,387,411,536,1325
0.8125,0.9792,578,529,649,1741
0.8542,0.9792,645,445,537,1602
0.8958,0.9792,647,336,422,1371
0.9375,0.9792,756,307,369,1384
0.9792,0.9792,694,237,258,1140
0.0208,0.9375,734,262,260,1208
0.0625,0.9375,792,375,316,1445
0.1042,0.9375,846,593,413,1838
0.1458,0.9375,1013,933,550,2507
0.1875,0.9375,974,1064,565,2628
0.2292,0.9375,697,859,483,2056
0.2708,0.9375,536,749,443,1740
0.3125,0.9375,469,699,423,1600
0.3542,0.9375,399,607,383,1394
0.3958,0.9375,412,636,448,1502
0.4375,0.9375,430,683,568,1686
0.4792,0.9375,457,743,722,1926
0.5208,0.9375,455,738,807,2001
0.5625,0.9375,321,550,667,1531
0.6042,0.9375,149,324,477,941
0.6458,0.9375,136,298,453,881
0.6875,0.9375,139,279,430,842
0.7292,0.9375,243,343,481,1060
0.7708,0.9375,395,406,521,1311
0.8125,0.9375,548,487,593,1615
0.8542,0.9375,636,376,461,1442
0.8958,0.9375,653,330,407,1354
0.9375,0.9375,668,269,316,1212
0.9792,0.9375,656,228,252,1091
0.0208,0.8958,769,287,280,1285
0.0625,0.8958,821,411,341,1537
0.1042,0.8958,932,669,463,2048
0.1458,0.8958,998,915,539,2459
0.1875,0.8958,863,929,504,2316
0.2292,0.8958,651,787,445,1896
0.2708,0.8958,537,742,439,1730
0.3125,0.8958,468,695,423,1594
0.3542,0.8958,395,598,378,1374
0.3958,0.8958,415,641,455,1515
0.4375,0.8958,424,662,553,1644
0.4792,0.8958,449,726,709,1887
0.5208,0.8958,466,749,817,2033
0.5625,0.8958,366,601,707,1671
0.6042,0.8958,248,451,597,1290
0.6458,0.8958,145,315,472,925
0.6875,0.8958,140,282,433,847
0.7292,0.8958,234,320,443,989
0.7708,0.8958,381,395,506,1272
0.8125,0.8958,561,502,608,1665
0.8542,0.8958,798,586,704,2061
0.8958,0.8958,709,392,484,1551
0.9375,0.8958,688,296,353,1296
0.9792,0.8958,613,228,250,1048
0.0208,0.8542,659,259,249,1126
0.0625,0.8542,729,382,313,1394
0.1042,0.8542,824,589,415,1813
0.1458,0.8542,882,805,481,2182
0.1875,0.8542,898,965,523,2408
0.2292,0.8542,680,824,468,1987
0.2708,0.8542,560,771,459,1801
0.3125,0.8542,475,709,436,1629
0.3542,0.8542,407,619,399,1431
0.3958,0.8542,430,667,485,1589
0.4375,0.8542,497,798,672,1975
0.4792,0.8542,501,823,808,2137
0.5208,0.8542,468,756,826,2051
0.5625,0.8542,407,671,781,1858
0.6042,0.8542,296,526,681,1498
0.6458,0.8542,167,347,509,1015
0.6875,0.8542,155,305,460,913
0.7292,0.8542,250,338,462,1044
0.7708,0.8542,375,390,494,1250
0.8125,0.8542,501,444,535,1466
0.8542,0.8542,633,452,537,1600
0.8958,0.8542,772,439,538,1713
0.9375,0.8542,722,327,388,1397
0.9792,0.8542,646,247,272,1124
0.0208,0.8125,695,288,279,1223
0.0625,0.8125,728,407,332,1441
0.1042,0.8125,789,587,406,1773
0.1458,0.8125,891,833,503,2234
0.1875,0.8125,889,959,529,2397
0.2292,0.8125,711,860,491,2078
0.2708,0.8125,530,720,438,1698
0.3125,0.8125,484,708,441,1643
0.3542,0.8125,452,696,453,1609
0.3958,0.8125,445,689,509,1647
0.4375,0.8125,463,732,623,1823
0.4792,0.8125,475,774,761,2014
0.5208,0.8125,476,764,830,2073
0.5625,0.8125,453,753,874,2080
0.6042,0.8125,337,593,756,1682
0.6458,0.8125,192,384,552,1121
0.6875,0.8125,183,343,505,1025
0.7292,0.8125,281,379,517,1170
0.7708,0.8125,455,481,605,1531
0.8125,0.8125,555,518,630,1692
0.8542,0.8125,643,475,562,1659
0.8958,0.8125,654,404,488,1519
0.9375,0.8125,822,439,525,1746
0.9792,0.8125,663,268,295,1185
0.0208,0.7708,710,325,311,1307
0.0625,0.7708,772,462,378,1588
0.1042,0.7708,805,598,431,1819
0.1458,0.7708,1019,980,604,2616
0.1875,0.7708,924,967,562,2470
0.2292,0.7708,802,964,558,2344
0.2708,0.7708,622,846,518,2001
0.3125,0.7708,514,761,480,1766
0.3542,0.7708,447,687,457,1600
0.3958,0.7708,453,693,516,1669
0.4375,0.7708,507,803,689,2007
0.4792,0.7708,491,790,774,2062
0.5208,0.7708,466,741,789,1999
0.5625,0.7708,419,678,776,1871
0.6042,0.7708,333,554,690,1573
0.6458,0.7708,209,390,541,1133
0.6875,0.7708,194,350,503,1041
0.7292,0.7708,298,392,518,1201
0.7708,0.7708,423,439,544,1396
0.8125,0.7708,575,527,626,1713
0.8542,0.7708,647,490,576,1694
0.8958,0.7708,651,394,473,1490
0.9375,0.7708,808,408,472,1643
0.9792,0.7708,698,306,331,1297
0.0208,0.7292,740,360,341,1405
0.0625,0.7292,717,454,372,1523
0.1042,0.7292,824,658,465,1939
0.1458,0.7292,932,898,564,2405
0.1875,0.7292,1013,1115,636,2791
0.2292,0.7292,793,957,570,2341
0.2708,0.7292,558,740,461,1770
0.3125,0.7292,515,756,484,1764
0.3542,0.7292,444,673,456,1581
0.3958,0.7292,448,687,523,1663
0.4375,0.7292,460,713,615,1793
0.4792,0.7292,516,826,810,2157
0.5208,0.7292,509,820,884,2216
0.5625,0.7292,402,640,726,1767
0.6042,0.7292,326,534,655,1510
0.6458,0.7292,241,422,570,1227
0.6875,0.7292,236,396,548,1174
0.7292,0.7292,347,446,578,1364
0.7708,0.7292,458,476,581,1505
0.8125,0.7292,541,493,578,1600
0.8542,0.7292,637,485,559,1662
0.8958,0.7292,690,431,508,1601
0.9375,0.7292,667,349,399,1382
0.9792,0.7292,728,344,365,1398
0.0208,0.6875,727,373,350,1416
0.0625,0.6875,744,462,392,1569
0.1042,0.6875,824,655,474,1943
0.1458,0.6875,903,879,558,2353
0.1875,0.6875,920,1008,596,2546
0.2292,0.6875,758,907,550,2233
0.2708,0.6875,581,775,488,1854
0.3125,0.6875,529,762,495,1795
0.3542,0.6875,438,652,452,1549
0.3958,0.6875,455,686,523,1670
0.4375,0.6875,493,760,658,1916
0.4792,0.6875,548,883,863,2300
0.5208,0.6875,494,777,834,2108
0.5625,0.6875,430,674,757,1861
0.6042,0.6875,345,548,661,1549
0.6458,0.6875,259,433,569,1255
0.6875,0.6875,248,392,530,1164
0.7292,0.6875,343,425,539,1298
0.7708,0.6875,471,481,581,1524
0.8125,0.6875,574,525,610,1696
0.8542,0.6875,688,527,603,1799
0.8958,0.6875,724,474,550,1720
0.9375,0.6875,711,392,440,1511
0.9792,0.6875,713,356,376,1411
0.0208,0.6458,873,500,464,1802
0.0625,0.6458,940,670,544,2134
0.1042,0.6458,969,829,598,2394
0.1458,0.6458,1055,1043,680,2793
0.1875,0.6458,1024,1133,687,2871
0.2292,0.6458,888,1080,670,2660
0.2708,0.6458,702,946,607,2270
0.3125,0.6458,555,794,524,1885
0.3542,0.6458,478,717,509,1712
0.3958,0.6458,465,696,543,1712
0.4375,0.6458,518,804,706,2035
0.4792,0.6458,517,814,795,2132
0.5208,0.6458,509,792,847,2150
0.5625,0.6458,442,688,764,1894
0.6042,0.6458,365,571,678,1611
0.6458,0.6458,319,513,654,1480
0.6875,0.6458,277,414,541,1227
0.7292,0.6458,366,452,563,1372
0.7708,0.6458,510,534,636,1671
0.8125,0.6458,568,521,596,1674
0.8542,0.6458,720,567,644,1912
0.8958,0.6458,699,474,538,1687
0.9375,0.6458,675,391,432,1470
0.9792,0.6458,719,381,396,1464
0.0208,0.6042,805,497,455,1730
0.0625,0.6042,796,591,484,1856
0.1042,0.6042,888,768,565,2219
0.1458,0.6042,1031,1046,691,2782
0.1875,0.6042,1083,1228,760,3098
0.2292,0.6042,809,975,626,2431
0.2708,0.6042,662,869,574,2118
0.3125,0.6042,615,886,596,2108
0.3542,0.6042,545,815,588,1958
0.3958,0.6042,541,803,633,1987
0.4375,0.6042,510,771,680,1968
0.4792,0.6042,561,871,853,2292
0.5208,0.6042,542,834,883,2265
0.5625,0.6042,464,702,775,1943
0.6042,0.6042,400,614,716,1727
0.6458,0.6042,330,506,629,1460
0.6875,0.6042,325,463,586,1368
0.7292,0.6042,418,517,630,1559
0.7708,0.6042,543,577,677,1788
0.8125,0.6042,622,588,667,1867
0.8542,0.6042,702,570,639,1895
0.8958,0.6042,709,504,560,1752
0.9375,0.6042,717,454,489,1635
0.9792,0.6042,846,495,500,1807
0.0208,0.5625,857,560,509,1900
0.0625,0.5625,866,662,543,2059
0.1042,0.5625,914,814,604,2331
0.1458,0.5625,982,990,670,2658
0.1875,0.5625,933,1042,667,2664
0.2292,0.5625,838,998,653,2509
0.2708,0.5625,723,949,641,2330
0.3125,0.5625,605,858,593,2069
0.3542,0.5625,560,834,609,2013
0.3958,0.5625,632,969,773,2385
0.4375,0.5625,640,984,873,2508
0.4792,0.5625,615,958,934,2516
0.5208,0.5625,612,945,990,2553
0.5625,0.5625,543,830,902,2280
0.6042,0.5625,418,627,720,1763
0.6458,0.5625,382,574,694,1645
0.6875,0.5625,367,513,633,1508
0.7292,0.5625,443,545,652,1634
0.7708,0.5625,550,582,672,1795
0.8125,0.5625,638,605,676,1910
0.8542,0.5625,729,610,673,1996
0.8958,0.5625,764,566,619,1929
0.9375,0.5625,764,515,546,1802
0.9792,0.5625,776,489,487,1726
0.0208,0.5208,854,604,540,1977
0.0625,0.5208,841,670,551,2052
0.1042,0.5208,903,818,616,2337
0.1458,0.5208,947,959,665,2583
0.1875,0.5208,977,1091,719,2809
0.2292,0.5208,878,1047,698,2643
0.2708,0.5208,758,988,682,2445
0.3125,0.5208,672,940,660,2288
0.3542,0.5208,560,811,606,1987
0.3958,0.5208,548,796,646,1999
0.4375,0.5208,567,839,746,2158
0.4792,0.5208,572,864,843,2286
0.5208,0.5208,567,852,883,2306
0.5625,0.5208,514,764,819,2100
0.6042,0.5208,449,663,746,1857
0.6458,0.5208,408,588,690,1681
0.6875,0.5208,412,570,679,1657
0.7292,0.5208,463,556,649,1662
0.7708,0.5208,555,591,669,1807
0.8125,0.5208,666,633,698,1987
0.8542,0.5208,728,618,673,2004
0.8958,0.5208,752,585,628,1948
0.9375,0.5208,751,524,545,1798
0.9792,0.5208,798,534,520,1829
0.0208,0.4792,969,729,648,2327
0.0625,0.4792,978,829,680,2478
0.1042,0.4792,1022,981,737,2746
0.1458,0.4792,1094,1149,803,3064
0.1875,0.4792,1082,1244,831,3184
0.2292,0.4792,947,1134,783,2888
0.2708,0.4792,793,1029,725,2564
0.3125,0.4792,655,893,644,2207
0.3542,0.4792,662,967,729,2370
0.3958,0.4792,622,909,741,2285
0.4375,0.4792,620,914,817,2361
0.4792,0.4792,627,939,911,2487
0.5208,0.4792,586,868,896,2356
0.5625,0.4792,535,782,829,2150
0.6042,0.4792,477,694,770,1941
0.6458,0.4792,423,602,694,1715
0.6875,0.4792,448,604,708,1756
0.7292,0.4792,501,609,695,1799
0.7708,0.4792,648,705,785,2131
0.8125,0.4792,692,675,731,2090
0.8542,0.4792,731,637,686,2041
0.8958,0.4792,788,630,666,2068
0.9375,0.4792,810,602,619,2011
0.9792,0.4792,863,621,598,2060
0.0208,0.4375,905,699,623,2212
0.0625,0.4375,864,738,610,2206
0.1042,0.4375,919,889,681,2493
0.1458,0.4375,1028,1079,771,2895
0.1875,0.4375,1067,1218,837,3146
0.2292,0.4375,1000,1216,852,3093
0.2708,0.4375,857,1125,804,2805
0.3125,0.4375,733,1025,754,2528
0.3542,0.4375,661,951,729,2355
0.3958,0.4375,609,865,709,2193
0.4375,0.4375,603,869,783,2264
0.4792,0.4375,622,911,879,2420
0.5208,0.4375,598,881,897,2382
0.5625,0.4375,580,840,881,2304
0.6042,0.4375,604,900,981,2485
0.6458,0.4375,554,812,920,2284
0.6875,0.4375,549,739,844,2130
0.7292,0.4375,545,656,736,1933
0.7708,0.4375,614,668,731,2007
0.8125,0.4375,720,710,760,2183
0.8542,0.4375,815,737,780,2321
0.8958,0.4375,898,765,795,2443
0.9375,0.4375,900,718,719,2320
0.9792,0.4375,872,672,640,2165
0.0208,0.3958,907,757,668,2321
0.0625,0.3958,912,826,682,2419
0.1042,0.3958,874,859,669,2408
0.1458,0.3958,938,986,726,2663
0.1875,0.3958,962,1089,768,2841
0.2292,0.3958,912,1095,787,2815
0.2708,0.3958,726,907,667,2314
0.3125,0.3958,645,845,637,2139
0.3542,0.3958,626,876,689,2202
0.3958,0.3958,693,995,831,2531
0.4375,0.3958,704,1019,915,2650
0.4792,0.3958,684,995,946,2635
0.5208,0.3958,656,957,962,2585
0.5625,0.3958,592,843,871,2310
0.6042,0.3958,533,751,810,2095
0.6458,0.3958,481,681,769,1929
0.6875,0.3958,482,640,729,1847
0.7292,0.3958,573,692,771,2032
0.7708,0.3958,648,717,784,2144
0.8125,0.3958,678,667,714,2052
0.8542,0.3958,762,705,735,2193
0.8958,0.3958,802,697,715,2203
0.9375,0.3958,804,657,652,2101
0.9792,0.3958,822,655,616,2079
0.0208,0.3542,912,790,697,2392
0.0625,0.3542,885,812,682,2377
0.1042,0.3542,909,921,726,2564
0.1458,0.3542,952,1015,765,2748
0.1875,0.3542,939,1063,775,2798
0.2292,0.3542,853,1002,749,2621
0.2708,0.3542,849,1073,810,2750
0.3125,0.3542,726,954,738,2434
0.3542,0.3542,655,892,710,2270
0.3958,0.3542,655,896,758,2319
0.4375,0.3542,668,936,843,2457
0.4792,0.3542,688,978,926,2603
0.5208,0.3542,668,960,953,2590
0.5625,0.3542,652,910,926,2494
0.6042,0.3542,599,828,865,2295
0.6458,0.3542,542,725,778,2045
0.6875,0.3542,525,670,733,1927
0.7292,0.3542,587,698,751,2034
0.7708,0.3542,655,715,755,2121
0.8125,0.3542,734,743,773,2245
0.8542,0.3542,809,771,790,2365
0.8958,0.3542,835,757,761,2345
0.9375,0.3542,849,738,721,2300
0.9792,0.3542,863,732,685,2269
0.0208,0.3125,967,890,780,2631
0.0625,0.3125,948,921,773,2644
0.1042,0.3125,1004,1054,842,2910
0.1458,0.3125,1135,1255,961,3371
0.1875,0.3125,1120,1315,987,3450
0.2292,0.3125,896,1061,804,2784
0.2708,0.3125,828,1028,793,2667
0.3125,0.3125,767,991,777,2550
0.3542,0.3125,714,946,766,2440
0.3958,0.3125,709,946,800,2469
0.4375,0.3125,704,960,870,2545
0.4792,0.3125,701,975,918,2607
0.5208,0.3125,662,926,910,2506
0.5625,0.3125,671,927,934,2539
0.6042,0.3125,649,880,910,2444
0.6458,0.3125,606,799,840,2249
0.6875,0.3125,586,746,796,2129
0.7292,0.3125,670,797,835,2302
0.7708,0.3125,765,845,872,2481
0.8125,0.3125,854,897,917,2667
0.8542,0.3125,923,919,925,2763
0.8958,0.3125,951,905,897,2747
0.9375,0.3125,865,782,755,2396
0.9792,0.3125,904,816,762,2475
0.0208,0.2708,910,853,754,2515
0.0625,0.2708,910,908,766,2589
0.1042,0.2708,964,1014,827,2816
0.1458,0.2708,1031,1130,889,3063
0.1875,0.2708,1139,1336,1026,3527
0.2292,0.2708,1043,1244,962,3276
0.2708,0.2708,918,1127,877,2944
0.3125,0.2708,906,1167,929,3021
0.3542,0.2708,872,1167,946,3003
0.3958,0.2708,878,1197,1024,3118
0.4375,0.2708,808,1103,990,2918
0.4792,0.2708,788,1078,998,2880
0.5208,0.2708,753,1032,997,2795
0.5625,0.2708,715,961,947,2632
0.6042,0.2708,695,922,934,2559
0.6458,0.2708,651,852,881,2388
0.6875,0.2708,646,806,839,2293
0.7292,0.2708,721,863,886,2474
0.7708,0.2708,743,823,836,2402
0.8125,0.2708,800,848,853,2500
0.8542,0.2708,834,834,827,2493
0.8958,0.2708,848,801,788,2432
0.9375,0.2708,900,824,795,2513
0.9792,0.2708,879,790,738,2400
0.0208,0.2292,1052,1065,937,3055
0.0625,0.2292,852,858,737,2451
0.1042,0.2292,971,1051,877,2914
0.1458,0.2292,991,1093,886,2988
0.1875,0.2292,990,1142,915,3069
0.2292,0.2292,910,1072,862,2863
0.2708,0.2292,896,1093,884,2891
0.3125,0.2292,826,1041,851,2735
0.3542,0.2292,804,1043,871,2733
0.3958,0.2292,837,1103,956,2911
0.4375,0.2292,870,1158,1036,3081
0.4792,0.2292,891,1211,1110,3229
0.5208,0.2292,877,1207,1144,3245
0.5625,0.2292,848,1147,1111,3118
0.6042,0.2292,828,1098,1088,3024
0.6458,0.2292,808,1055,1064,2934
0.6875,0.2292,810,1025,1038,2879
0.7292,0.2292,835,1003,1003,2847
0.7708,0.2292,904,1036,1025,2970
0.8125,0.2292,971,1088,1070,3135
0.8542,0.2292,895,944,922,2763
0.8958,0.2292,899,918,884,2702
0.9375,0.2292,860,850,796,2507
0.9792,0.2292,843,817,754,2413
0.0208,0.1875,904,927,820,2656
0.0625,0.1875,993,1071,924,2996
0.1042,0.1875,1134,1283,1072,3507
0.1458,0.1875,982,1098,907,3006
0.1875,0.1875,1011,1174,958,3164
0.2292,0.1875,977,1167,958,3125
0.2708,0.1875,907,1109,914,2948
0.3125,0.1875,960,1215,1011,3207
0.3542,0.1875,922,1200,1018,3159
0.3958,0.1875,940,1233,1076,3269
0.4375,0.1875,878,1152,1028,3076
0.4792,0.1875,877,1163,1063,3120
0.5208,0.1875,825,1082,1013,2933
0.5625,0.1875,805,1036,990,2843
0.6042,0.1875,764,971,947,2692
0.6458,0.1875,692,857,850,2405
0.6875,0.1875,763,933,925,2627
0.7292,0.1875,824,978,961,2772
0.7708,0.1875,816,933,909,2664
0.8125,0.1875,857,950,917,2729
0.8542,0.1875,868,932,894,2699
0.8958,0.1875,866,914,863,2649
0.9375,0.1875,898,923,852,2676
0.9792,0.1875,891,913,830,2639
0.0208,0.1458,908,972,858,2747
0.0625,0.1458,987,1089,949,3040
0.1042,0.1458,972,1089,929,3006
0.1458,0.1458,1021,1159,981,3181
0.1875,0.1458,1067,1252,1045,3389
0.2292,0.1458,968,1165,985,3138
0.2708,0.1458,916,1105,940,2980
0.3125,0.1458,896,1104,940,2959
0.3542,0.1458,810,1001,872,2698
0.3958,0.1458,827,1035,910,2788
0.4375,0.1458,847,1067,959,2888
0.4792,0.1458,820,1040,944,2820
0.5208,0.1458,815,1028,951,2807
0.5625,0.1458,787,993,920,2712
0.6042,0.1458,796,986,943,2736
0.6458,0.1458,792,967,929,2698
0.6875,0.1458,801,963,934,2706
0.7292,0.1458,822,967,931,2728
0.7708,0.1458,796,901,863,2575
0.8125,0.1458,851,951,908,2718
0.8542,0.1458,852,926,876,2661
0.8958,0.1458,957,1044,974,2982
0.9375,0.1458,912,973,891,2783
0.9792,0.1458,889,943,848,2688
0.0208,0.1042,1105,1236,1093,3451
0.0625,0.1042,1099,1249,1095,3463
0.1042,0.1042,1093,1266,1099,3478
0.1458,0.1042,888,1013,876,2794
0.1875,0.1042,1119,1324,1131,3600
0.2292,0.1042,1034,1242,1063,3361
0.2708,0.1042,1071,1303,1122,3519
0.3125,0.1042,973,1199,1040,3234
0.3542,0.1042,1012,1270,1112,3415
0.3958,0.1042,976,1225,1089,3310
0.4375,0.1042,968,1220,1098,3308
0.4792,0.1042,938,1180,1084,3221
0.5208,0.1042,925,1150,1068,3161
0.5625,0.1042,860,1046,982,2903
0.6042,0.1042,840,1008,951,2811
0.6458,0.1042,872,1031,979,2896
0.6875,0.1042,862,1005,955,2836
0.7292,0.1042,899,1038,976,2926
0.7708,0.1042,906,1013,948,2880
0.8125,0.1042,896,994,927,2832
0.8542,0.1042,882,967,891,2752
0.8958,0.1042,907,995,909,2823
0.9375,0.1042,915,1000,900,2827
0.9792,0.1042,1066,1192,1067,3340
0.0208,0.0625,1083,1253,1112,3467
0.0625,0.0625,690,762,684,2146
0.1042,0.0625,693,776,689,2167
0.1458,0.0625,1134,1339,1168,3666
0.1875,0.0625,1118,1330,1159,3633
0.2292,0.0625,1072,1289,1130,3519
0.2708,0.0625,1068,1302,1145,3540
0.3125,0.0625,1027,1260,1113,3428
0.3542,0.0625,1049,1284,1139,3496
0.3958,0.0625,1029,1266,1130,3447
0.4375,0.0625,1028,1270,1141,3462
0.4792,0.0625,1006,1232,1124,3383
0.5208,0.0625,976,1192,1090,3279
0.5625,0.0625,936,1134,1049,3143
0.6042,0.0625,986,1195,1109,3309
0.6458,0.0625,1023,1227,1138,3408
0.6875,0.0625,980,1157,1075,3231
0.7292,0.0625,1003,1176,1091,3287
0.7708,0.0625,1037,1209,1116,3380
0.8125,0.0625,1044,1214,1112,3389
0.8542,0.0625,1072,1251,1129,3474
0.8958,0.0625,1080,1242,1115,3458
0.9375,0.0625,1087,1252,1124,3484
0.9792,0.0625,1088,1253,1125,3487
0.0208,0.0208,1117,1321,1190,3651
0.0625,0.0208,1102,1307,1171,3604
0.1042,0.0208,905,1042,940,2907
0.1458,0.0208,982,1158,1038,3201
0.1875,0.0208,913,1062,954,2948
0.2292,0.0208,933,1098,985,3029
0.2708,0.0208,949,1118,1003,3082
0.3125,0.0208,730,893,797,2429
0.3542,0.0208,554,637,580,1780
0.3958,0.0208,489,561,511,1567
0.4375,0.0208,476,550,499,1530
0.4792,0.0208,505,589,535,1634
0.5208,0.0208,809,1010,907,2733
0.5625,0.0208,636,769,696,2117
0.6042,0.0208,709,844,770,2342
0.6458,0.0208,767,944,852,2579
0.6875,0.0208,636,778,702,2127
0.7292,0.0208,934,1086,991,3026
0.7708,0.0208,961,1122,1024,3125
0.8125,0.0208,874,1023,935,2856
0.8542,0.0208,912,1053,953,2935
0.8958,0.0208,940,1089,985,3033
0.9375,0.0208,930,1075,973,2994
0.9792,0.0208,1027,1202,1087,3338
"""

In [ ]:
PACIFIC_BLUE_VER = """
0.05,0.95,757,296,265,1271
0.15,0.95,1013,924,513,2460
0.25,0.95,643,866,483,2004
0.35,0.95,427,644,386,1466
0.45,0.95,494,809,687,1997
0.55,0.95,401,680,802,1881
0.65,0.95,117,280,450,838
0.75,0.95,308,368,507,1173
0.85,0.95,723,367,437,1478
0.95,0.95,725,270,314,1257
0.05,0.85,859,391,338,1541
0.15,0.85,1085,1021,579,2696
0.25,0.85,693,914,510,2135
0.35,0.85,471,747,461,1687
0.45,0.85,477,786,684,1954
0.55,0.85,473,822,959,2251
0.65,0.85,164,351,532,1040
0.75,0.85,338,395,526,1250
0.85,0.85,774,581,697,2025
0.95,0.85,732,301,349,1334
0.05,0.75,810,431,367,1574
0.15,0.75,983,929,556,2479
0.25,0.75,668,877,514,2073
0.35,0.75,490,766,491,1755
0.45,0.75,533,867,761,2169
0.55,0.75,482,790,905,2177
0.65,0.75,205,385,550,1133
0.75,0.75,398,458,590,1437
0.85,0.75,677,512,599,1768
0.95,0.75,807,378,427,1565
0.05,0.65,871,545,459,1847
0.15,0.65,1040,1031,639,2727
0.25,0.65,712,908,563,2198
0.35,0.65,499,771,521,1798
0.45,0.65,584,943,849,2385
0.55,0.65,459,724,809,1991
0.65,0.65,261,422,556,1233
0.75,0.65,431,490,606,1518
0.85,0.65,716,564,644,1906
0.95,0.65,817,442,479,1699
0.05,0.55,931,671,562,2143
0.15,0.55,1065,1068,704,2853
0.25,0.55,892,1128,727,2767
0.35,0.55,583,871,621,2086
0.45,0.55,591,918,838,2355
0.55,0.55,540,829,908,2278
0.65,0.55,360,532,655,1542
0.75,0.55,495,557,658,1701
0.85,0.55,724,604,668,1980
0.95,0.55,818,532,551,1873
0.05,0.45,1043,853,709,2592
0.15,0.45,1163,1221,844,3250
0.25,0.45,902,1127,783,2832
0.35,0.45,672,967,719,2370
0.45,0.45,605,886,819,2317
0.55,0.45,558,820,866,2248
0.65,0.45,460,651,756,1864
0.75,0.45,549,625,706,1875
0.85,0.45,716,623,669,1997
0.95,0.45,862,639,637,2117
0.05,0.35,987,885,746,2612
0.15,0.35,1022,1098,802,2934
0.25,0.35,931,1135,834,2920
0.35,0.35,674,923,725,2333
0.45,0.35,629,881,808,2330
0.55,0.35,671,960,981,2618
0.65,0.35,562,762,830,2154
0.75,0.35,634,719,771,2119
0.85,0.35,802,763,785,2344
0.95,0.35,907,766,736,2396
0.05,0.25,1066,1063,904,3037
0.15,0.25,1183,1319,1025,3549
0.25,0.25,1067,1312,1015,3421
0.35,0.25,880,1162,942,3003
0.45,0.25,816,1115,1014,2958
0.55,0.25,721,978,966,2675
0.65,0.25,668,868,899,2438
0.75,0.25,779,911,929,2622
0.85,0.25,883,902,895,2678
0.95,0.25,931,871,815,2611
0.05,0.15,1093,1190,1036,3332
0.15,0.15,1155,1322,1090,3590
0.25,0.15,965,1156,965,3104
0.35,0.15,923,1179,1013,3132
0.45,0.15,925,1189,1070,3202
0.55,0.15,859,1102,1035,3012
0.65,0.15,845,1052,1028,2937
0.75,0.15,948,1120,1084,3162
0.85,0.15,1048,1167,1113,3338
0.95,0.15,1032,1098,998,3137
0.05,0.05,1236,1456,1285,4002
0.15,0.05,1193,1414,1238,3872
0.25,0.05,1198,1457,1277,3959
0.35,0.05,1157,1441,1275,3899
0.45,0.05,1145,1422,1281,3875
0.55,0.05,1146,1413,1289,3872
0.65,0.05,1121,1372,1266,3780
0.75,0.05,1115,1342,1241,3718
0.85,0.05,1142,1332,1221,3716
0.95,0.05,1156,1344,1211,3732
"""

In [ ]:
REDWOOD_CAL = """
0.0208,0.9792,784,265,260,1243
0.0625,0.9792,855,384,320,1500
0.1042,0.9792,910,632,430,1937
0.1458,0.9792,998,900,518,2401
0.1875,0.9792,956,1039,543,2565
0.2292,0.9792,723,897,496,2113
0.2708,0.9792,579,812,474,1860
0.3125,0.9792,494,734,439,1661
0.3542,0.9792,472,719,444,1628
0.3958,0.9792,451,692,474,1608
0.4375,0.9792,460,730,595,1774
0.4792,0.9792,503,816,788,2091
0.5208,0.9792,516,834,908,2240
0.5625,0.9792,418,697,825,1919
0.6042,0.9792,253,488,667,1389
0.6458,0.9792,152,355,548,1038
0.6875,0.9792,147,315,499,946
0.7292,0.9792,248,357,506,1093
0.7708,0.9792,424,447,579,1425
0.8125,0.9792,662,603,734,1963
0.8542,0.9792,917,659,789,2312
0.8958,0.9792,903,478,596,1913
0.9375,0.9792,758,307,364,1369
0.9792,0.9792,703,250,280,1174
0.0208,0.9375,834,299,288,1353
0.0625,0.9375,922,426,352,1639
0.1042,0.9375,958,672,455,2049
0.1458,0.9375,1141,1034,600,2759
0.1875,0.9375,1074,1171,614,2860
0.2292,0.9375,853,1063,586,2501
0.2708,0.9375,678,966,556,2196
0.3125,0.9375,610,901,533,2039
0.3542,0.9375,512,793,492,1790
0.3958,0.9375,515,806,556,1869
0.4375,0.9375,569,917,751,2226
0.4792,0.9375,622,1040,995,2642
0.5208,0.9375,614,1015,1106,2714
0.5625,0.9375,485,825,976,2264
0.6042,0.9375,298,570,760,1607
0.6458,0.9375,182,427,656,1246
0.6875,0.9375,169,360,562,1075
0.7292,0.9375,297,426,599,1303
0.7708,0.9375,471,497,640,1582
0.8125,0.9375,628,573,695,1863
0.8542,0.9375,836,607,725,2120
0.8958,0.9375,844,520,631,1936
0.9375,0.9375,809,438,539,1729
0.9792,0.9375,756,266,290,1249
0.0208,0.8958,870,331,317,1449
0.0625,0.8958,867,402,344,1557
0.1042,0.8958,987,698,479,2128
0.1458,0.8958,1165,1066,620,2836
0.1875,0.8958,1064,1138,609,2811
0.2292,0.8958,882,1069,582,2531
0.2708,0.8958,687,972,565,2222
0.3125,0.8958,575,864,518,1952
0.3542,0.8958,501,769,480,1745
0.3958,0.8958,497,773,515,1782
0.4375,0.8958,545,866,711,2112
0.4792,0.8958,565,915,855,2322
0.5208,0.8958,619,1016,1106,2720
0.5625,0.8958,474,792,927,2173
0.6042,0.8958,336,634,840,1788
0.6458,0.8958,193,421,624,1220
0.6875,0.8958,182,384,590,1138
0.7292,0.8958,297,413,573,1264
0.7708,0.8958,455,480,613,1522
0.8125,0.8958,508,513,643,1639
0.8542,0.8958,661,598,721,1947
0.8958,0.8958,916,665,793,2320
0.9375,0.8958,936,527,646,2044
0.9792,0.8958,753,321,378,1394
0.0208,0.8542,800,308,294,1341
0.0625,0.8542,862,455,371,1637
0.1042,0.8542,936,689,473,2065
0.1458,0.8542,1026,950,558,2521
0.1875,0.8542,1116,1237,664,3015
0.2292,0.8542,684,890,518,2085
0.2708,0.8542,625,856,505,1981
0.3125,0.8542,563,835,507,1898
0.3542,0.8542,491,751,478,1713
0.3958,0.8542,521,825,607,1945
0.4375,0.8542,549,878,737,2152
0.4792,0.8542,624,1036,1010,2655
0.5208,0.8542,668,1123,1218,2990
0.5625,0.8542,519,876,1015,2389
0.6042,0.8542,336,596,770,1681
0.6458,0.8542,205,427,623,1238
0.6875,0.8542,195,389,588,1154
0.7292,0.8542,324,444,606,1354
0.7708,0.8542,472,498,627,1571
0.8125,0.8542,622,557,669,1815
0.8542,0.8542,787,572,675,1990
0.8958,0.8542,959,548,668,2110
0.9375,0.8542,803,364,425,1531
0.9792,0.8542,765,293,319,1315
0.0208,0.8125,812,434,403,1594
0.0625,0.8125,980,518,442,1885
0.1042,0.8125,1080,679,517,2230
0.1458,0.8125,1164,1029,654,2834
0.1875,0.8125,1312,1391,769,3474
0.2292,0.8125,966,1141,636,2740
0.2708,0.8125,632,859,513,1998
0.3125,0.8125,600,891,547,2033
0.3542,0.8125,542,839,538,1913
0.3958,0.8125,503,777,567,1839
0.4375,0.8125,512,804,663,1969
0.4792,0.8125,526,850,826,2187
0.5208,0.8125,525,837,893,2239
0.5625,0.8125,557,896,972,2405
0.6042,0.8125,458,755,870,2062
0.6458,0.8125,324,546,692,1542
0.6875,0.8125,214,421,602,1219
0.7292,0.8125,196,365,538,1082
0.7708,0.8125,330,433,578,1320
0.8125,0.8125,476,491,613,1554
0.8542,0.8125,627,566,673,1833
0.8958,0.8125,752,546,642,1897
0.9375,0.8125,1003,595,717,2250
0.9792,0.8125,865,410,475,1684
0.0208,0.7708,944,433,409,1719
0.0625,0.7708,949,574,465,1937
0.1042,0.7708,997,772,534,2271
0.1458,0.7708,1046,989,605,2626
0.1875,0.7708,1016,1104,617,2736
0.2292,0.7708,846,1009,580,2432
0.2708,0.7708,674,912,556,2138
0.3125,0.7708,643,966,601,2205
0.3542,0.7708,521,797,525,1835
0.3958,0.7708,486,736,522,1738
0.4375,0.7708,557,866,639,2053
0.4792,0.7708,580,935,800,2304
0.5208,0.7708,583,938,986,2490
0.5625,0.7708,564,927,1054,2522
0.6042,0.7708,397,666,819,1861
0.6458,0.7708,296,519,673,1467
0.6875,0.7708,246,455,635,1315
0.7292,0.7708,328,485,661,1455
0.7708,0.7708,536,573,714,1797
0.8125,0.7708,601,619,756,1947
0.8542,0.7708,785,617,721,2082
0.8958,0.7708,765,512,603,1833
0.9375,0.7708,821,410,471,1645
0.9792,0.7708,952,429,461,1770
0.0208,0.7292,888,457,423,1711
0.0625,0.7292,860,541,436,1796
0.1042,0.7292,898,680,490,2038
0.1458,0.7292,1021,976,612,2596
0.1875,0.7292,1068,1173,663,2906
0.2292,0.7292,948,1156,678,2783
0.2708,0.7292,712,959,589,2256
0.3125,0.7292,709,1065,671,2441
0.3542,0.7292,676,1073,711,2454
0.3958,0.7292,635,1004,754,2383
0.4375,0.7292,656,1047,895,2586
0.4792,0.7292,567,907,882,2340
0.5208,0.7292,615,989,1062,2648
0.5625,0.7292,617,1025,1155,2774
0.6042,0.7292,455,753,892,2075
0.6458,0.7292,285,491,637,1387
0.6875,0.7292,269,469,637,1356
0.7292,0.7292,284,463,630,1353
0.7708,0.7292,507,534,651,1666
0.8125,0.7292,632,586,688,1875
0.8542,0.7292,756,582,669,1967
0.8958,0.7292,778,513,597,1841
0.9375,0.7292,754,398,451,1553
0.9792,0.7292,970,463,494,1858
0.0208,0.6875,917,490,453,1804
0.0625,0.6875,980,686,551,2176
0.1042,0.6875,1029,872,616,2491
0.1458,0.6875,1109,1096,703,2898
0.1875,0.6875,1091,1209,719,3021
0.2292,0.6875,912,1104,671,2686
0.2708,0.6875,803,1088,689,2578
0.3125,0.6875,746,1115,722,2580
0.3542,0.6875,557,844,591,1985
0.3958,0.6875,571,864,669,2095
0.4375,0.6875,651,1017,884,2540
0.4792,0.6875,580,911,883,2360
0.5208,0.6875,671,1060,1124,2837
0.5625,0.6875,655,1063,1174,2869
0.6042,0.6875,459,731,865,2034
0.6458,0.6875,346,548,693,1567
0.6875,0.6875,327,489,634,1431
0.7292,0.6875,420,522,648,1568
0.7708,0.6875,540,563,665,1741
0.8125,0.6875,700,649,738,2054
0.8542,0.6875,868,710,800,2335
0.8958,0.6875,895,632,709,2185
0.9375,0.6875,798,482,525,1754
0.9792,0.6875,901,487,500,1831
0.0208,0.6458,1064,623,567,2193
0.0625,0.6458,1205,874,703,2733
0.1042,0.6458,1253,1061,769,3050
0.1458,0.6458,1338,1351,866,3544
0.1875,0.6458,1329,1506,897,3737
0.2292,0.6458,987,1192,733,2910
0.2708,0.6458,712,939,601,2247
0.3125,0.6458,658,947,618,2218
0.3542,0.6458,538,806,567,1903
0.3958,0.6458,580,875,677,2121
0.4375,0.6458,562,861,757,2167
0.4792,0.6458,597,933,924,2439
0.5208,0.6458,768,1244,1325,3313
0.5625,0.6458,533,853,960,2325
0.6042,0.6458,435,683,806,1904
0.6458,0.6458,338,534,676,1529
0.6875,0.6458,317,471,612,1381
0.7292,0.6458,429,532,659,1597
0.7708,0.6458,534,555,655,1717
0.8125,0.6458,676,627,710,1981
0.8542,0.6458,774,615,692,2042
0.8958,0.6458,763,523,588,1831
0.9375,0.6458,761,445,485,1644
0.9792,0.6458,898,488,499,1828
0.0208,0.6042,908,561,507,1927
0.0625,0.6042,959,714,578,2212
0.1042,0.6042,1060,918,667,2617
0.1458,0.6042,1100,1091,717,2893
0.1875,0.6042,1112,1229,758,3096
0.2292,0.6042,939,1127,714,2776
0.2708,0.6042,794,1043,680,2512
0.3125,0.6042,800,1152,765,2712
0.3542,0.6042,575,864,617,2049
0.3958,0.6042,575,853,665,2085
0.4375,0.6042,569,861,753,2172
0.4792,0.6042,591,915,888,2380
0.5208,0.6042,625,967,1015,2590
0.5625,0.6042,635,991,1083,2688
0.6042,0.6042,434,664,769,1848
0.6458,0.6042,357,544,670,1553
0.6875,0.6042,349,492,618,1439
0.7292,0.6042,441,542,656,1617
0.7708,0.6042,538,564,657,1733
0.8125,0.6042,651,615,691,1926
0.8542,0.6042,736,600,667,1967
0.8958,0.6042,751,531,581,1822
0.9375,0.6042,761,483,512,1712
0.9792,0.6042,893,527,527,1895
0.0208,0.5625,1054,708,634,2346
0.0625,0.5625,1020,792,640,2415
0.1042,0.5625,1139,1034,756,2904
0.1458,0.5625,1286,1338,892,3512
0.1875,0.5625,1157,1309,831,3300
0.2292,0.5625,1015,1222,789,3026
0.2708,0.5625,811,1059,708,2573
0.3125,0.5625,761,1075,737,2567
0.3542,0.5625,580,853,618,2043
0.3958,0.5625,531,771,614,1907
0.4375,0.5625,593,880,777,2237
0.4792,0.5625,650,1003,971,2609
0.5208,0.5625,626,942,980,2531
0.5625,0.5625,636,977,1052,2644
0.6042,0.5625,456,682,777,1895
0.6458,0.5625,366,535,641,1524
0.6875,0.5625,358,488,599,1426
0.7292,0.5625,429,519,616,1544
0.7708,0.5625,524,546,624,1670
0.8125,0.5625,626,587,650,1834
0.8542,0.5625,707,584,638,1896
0.8958,0.5625,752,557,602,1873
0.9375,0.5625,720,485,509,1675
0.9792,0.5625,724,451,445,1580
0.0208,0.5208,905,641,571,2076
0.0625,0.5208,989,791,644,2389
0.1042,0.5208,1015,922,688,2602
0.1458,0.5208,1155,1187,809,3139
0.1875,0.5208,1190,1340,873,3400
0.2292,0.5208,1023,1212,803,3031
0.2708,0.5208,851,1091,749,2685
0.3125,0.5208,669,911,637,2209
0.3542,0.5208,595,844,626,2056
0.3958,0.5208,665,967,776,2397
0.4375,0.5208,678,1008,888,2561
0.4792,0.5208,613,918,886,2403
0.5208,0.5208,680,1020,1049,2732
0.5625,0.5208,655,985,1047,2667
0.6042,0.5208,484,712,795,1971
0.6458,0.5208,449,641,747,1817
0.6875,0.5208,449,608,723,1758
0.7292,0.5208,475,572,655,1682
0.7708,0.5208,594,635,705,1913
0.8125,0.5208,760,750,814,2296
0.8542,0.5208,902,810,867,2544
0.8958,0.5208,980,796,841,2572
0.9375,0.5208,1028,754,772,2502
0.9792,0.5208,934,663,637,2188
0.0208,0.4792,949,702,619,2229
0.0625,0.4792,931,773,632,2305
0.1042,0.4792,906,852,638,2377
0.1458,0.4792,1011,1041,725,2767
0.1875,0.4792,998,1116,744,2854
0.2292,0.4792,942,1119,769,2825
0.2708,0.4792,872,1120,782,2769
0.3125,0.4792,672,911,653,2230
0.3542,0.4792,644,911,686,2234
0.3958,0.4792,717,1042,842,2591
0.4375,0.4792,728,1067,947,2728
0.4792,0.4792,614,902,868,2371
0.5208,0.4792,683,1004,1028,2698
0.5625,0.4792,775,1172,1229,3155
0.6042,0.4792,526,768,845,2119
0.6458,0.4792,437,618,707,1744
0.6875,0.4792,449,597,694,1721
0.7292,0.4792,503,603,682,1766
0.7708,0.4792,596,635,702,1908
0.8125,0.4792,688,665,713,2039
0.8542,0.4792,751,657,699,2074
0.8958,0.4792,803,654,680,2103
0.9375,0.4792,799,610,612,1986
0.9792,0.4792,888,653,618,2119
0.0208,0.4375,1060,841,737,2599
0.0625,0.4375,950,824,675,2422
0.1042,0.4375,918,883,673,2456
0.1458,0.4375,1026,1061,754,2831
0.1875,0.4375,1066,1194,819,3074
0.2292,0.4375,1146,1379,959,3480
0.2708,0.4375,1012,1316,932,3255
0.3125,0.4375,878,1214,887,2972
0.3542,0.4375,820,1178,896,2888
0.3958,0.4375,816,1182,959,2948
0.4375,0.4375,865,1292,1150,3296
0.4792,0.4375,806,1206,1152,3152
0.5208,0.4375,728,1075,1086,2874
0.5625,0.4375,842,1260,1309,3392
0.6042,0.4375,648,948,1024,2600
0.6458,0.4375,555,784,880,2198
0.6875,0.4375,591,786,889,2242
0.7292,0.4375,643,783,868,2270
0.7708,0.4375,731,807,877,2389
0.8125,0.4375,811,806,854,2442
0.8542,0.4375,836,758,793,2354
0.8958,0.4375,871,731,753,2318
0.9375,0.4375,853,669,665,2150
0.9792,0.4375,916,700,661,2237
0.0208,0.3958,1260,1078,936,3233
0.0625,0.3958,1162,1063,868,3064
0.1042,0.3958,989,967,755,2693
0.1458,0.3958,941,979,716,2625
0.1875,0.3958,990,1109,778,2871
0.2292,0.3958,1000,1181,845,3020
0.2708,0.3958,840,1049,764,2645
0.3125,0.3958,785,1035,770,2582
0.3542,0.3958,636,871,680,2178
0.3958,0.3958,778,1097,911,2776
0.4375,0.3958,742,1050,937,2716
0.4792,0.3958,713,1017,961,2677
0.5208,0.3958,850,1243,1237,3314
0.5625,0.3958,827,1211,1240,3259
0.6042,0.3958,680,965,1019,2644
0.6458,0.3958,549,749,819,2097
0.6875,0.3958,557,714,790,2039
0.7292,0.3958,589,715,771,2056
0.7708,0.3958,660,721,763,2119
0.8125,0.3958,726,736,767,2201
0.8542,0.3958,813,777,793,2358
0.8958,0.3958,919,837,835,2559
0.9375,0.3958,916,793,767,2442
0.9792,0.3958,957,806,751,2482
0.0208,0.3542,1237,1107,962,3271
0.0625,0.3542,1081,1024,845,2925
0.1042,0.3542,1020,1035,808,2846
0.1458,0.3542,1061,1135,849,3035
0.1875,0.3542,1096,1249,909,3249
0.2292,0.3542,1006,1189,879,3068
0.2708,0.3542,912,1136,852,2894
0.3125,0.3542,804,1050,806,2660
0.3542,0.3542,720,974,768,2453
0.3958,0.3542,720,975,817,2501
0.4375,0.3542,741,1030,922,2680
0.4792,0.3542,767,1085,1014,2853
0.5208,0.3542,824,1179,1159,3147
0.5625,0.3542,868,1237,1246,3333
0.6042,0.3542,745,1038,1075,2839
0.6458,0.3542,670,903,960,2513
0.6875,0.3542,661,857,926,2421
0.7292,0.3542,700,840,896,2413
0.7708,0.3542,764,837,870,2448
0.8125,0.3542,823,839,863,2498
0.8542,0.3542,975,944,956,2843
0.8958,0.3542,979,901,896,2743
0.9375,0.3542,909,797,771,2444
0.9792,0.3542,985,850,787,2588
0.0208,0.3125,1024,922,806,2721
0.0625,0.3125,1124,1076,901,3074
0.1042,0.3125,1142,1179,940,3243
0.1458,0.3125,1359,1501,1142,3993
0.1875,0.3125,1302,1536,1141,3981
0.2292,0.3125,1077,1295,968,3338
0.2708,0.3125,964,1202,917,3078
0.3125,0.3125,886,1148,890,2917
0.3542,0.3125,878,1185,945,3000
0.3958,0.3125,863,1172,978,3006
0.4375,0.3125,796,1091,976,2852
0.4792,0.3125,740,1021,950,2698
0.5208,0.3125,900,1280,1244,3412
0.5625,0.3125,895,1263,1252,3393
0.6042,0.3125,697,945,970,2594
0.6458,0.3125,667,879,916,2443
0.6875,0.3125,618,783,828,2209
0.7292,0.3125,685,810,841,2314
0.7708,0.3125,772,850,869,2467
0.8125,0.3125,827,861,872,2534
0.8542,0.3125,928,909,908,2715
0.8958,0.3125,939,880,864,2652
0.9375,0.3125,897,807,773,2447
0.9792,0.3125,1023,917,848,2755
0.0208,0.2708,1085,1029,900,2989
0.0625,0.2708,1110,1121,934,3144
0.1042,0.2708,1088,1148,925,3144
0.1458,0.2708,1168,1278,994,3428
0.1875,0.2708,1361,1589,1212,4159
0.2292,0.2708,1209,1451,1111,3768
0.2708,0.2708,1090,1355,1045,3486
0.3125,0.2708,943,1195,947,3078
0.3542,0.2708,883,1154,931,2959
0.3958,0.2708,861,1133,965,2947
0.4375,0.2708,790,1054,941,2772
0.4792,0.2708,755,1012,932,2687
0.5208,0.2708,763,1034,988,2770
0.5625,0.2708,730,973,948,2634
0.6042,0.2708,675,890,890,2439
0.6458,0.2708,660,853,871,2366
0.6875,0.2708,685,861,883,2408
0.7292,0.2708,738,880,892,2489
0.7708,0.2708,838,933,936,2683
0.8125,0.2708,913,973,967,2827
0.8542,0.2708,998,1016,996,2982
0.8958,0.2708,926,905,874,2678
0.9375,0.2708,949,904,850,2675
0.9792,0.2708,1051,980,896,2897
0.0208,0.2292,1410,1456,1266,4106
0.0625,0.2292,1205,1268,1071,3526
0.1042,0.2292,949,1021,846,2802
0.1458,0.2292,972,1062,855,2876
0.1875,0.2292,1008,1154,919,3073
0.2292,0.2292,985,1156,922,3054
0.2708,0.2292,996,1209,969,3166
0.3125,0.2292,851,1061,861,2764
0.3542,0.2292,804,1017,844,2655
0.3958,0.2292,783,1001,863,2636
0.4375,0.2292,790,1024,911,2715
0.4792,0.2292,864,1138,1037,3026
0.5208,0.2292,868,1164,1094,3113
0.5625,0.2292,818,1075,1034,2911
0.6042,0.2292,764,982,964,2694
0.6458,0.2292,737,926,921,2567
0.6875,0.2292,718,891,893,2490
0.7292,0.2292,762,903,890,2530
0.7708,0.2292,809,907,889,2560
0.8125,0.2292,871,952,926,2728
0.8542,0.2292,853,894,859,2584
0.8958,0.2292,899,914,870,2660
0.9375,0.2292,920,913,844,2653
0.9792,0.2292,1343,1365,1242,3921
0.0208,0.1875,1083,1116,981,3159
0.0625,0.1875,1439,1600,1362,4387
0.1042,0.1875,1195,1340,1115,3640
0.1458,0.1875,1104,1247,1022,3365
0.1875,0.1875,1034,1176,960,3162
0.2292,0.1875,1022,1177,956,3146
0.2708,0.1875,1030,1228,1004,3249
0.3125,0.1875,1055,1285,1056,3388
0.3542,0.1875,900,1135,955,2982
0.3958,0.1875,848,1074,934,2845
0.4375,0.1875,912,1166,1031,3097
0.4792,0.1875,922,1196,1086,3192
0.5208,0.1875,856,1106,1027,2976
0.5625,0.1875,843,1077,1020,2925
0.6042,0.1875,822,1043,1006,2855
0.6458,0.1875,749,927,909,2568
0.6875,0.1875,762,918,904,2566
0.7292,0.1875,808,947,921,2656
0.7708,0.1875,823,935,902,2640
0.8125,0.1875,852,939,899,2669
0.8542,0.1875,897,961,915,2752
0.8958,0.1875,870,918,859,2626
0.9375,0.1875,985,1008,932,2915
0.9792,0.1875,1130,1160,1042,3288
0.0208,0.1458,1050,1131,989,3152
0.0625,0.1458,1247,1377,1192,3800
0.1042,0.1458,1494,1756,1472,4717
0.1458,0.1458,1215,1412,1182,3802
0.1875,0.1458,1154,1356,1123,3626
0.2292,0.1458,1061,1262,1058,3372
0.2708,0.1458,1141,1412,1199,3753
0.3125,0.1458,1181,1490,1268,3930
0.3542,0.1458,1091,1392,1198,3671
0.3958,0.1458,1160,1490,1303,3944
0.4375,0.1458,1141,1465,1299,3897
0.4792,0.1458,1139,1470,1323,3922
0.5208,0.1458,1049,1353,1235,3624
0.5625,0.1458,1116,1437,1337,3879
0.6042,0.1458,1126,1439,1353,3905
0.6458,0.1458,1093,1381,1309,3770
0.6875,0.1458,1092,1359,1304,3739
0.7292,0.1458,1114,1350,1285,3732
0.7708,0.1458,1060,1231,1165,3438
0.8125,0.1458,1037,1174,1109,3301
0.8542,0.1458,1061,1175,1099,3316
0.8958,0.1458,1089,1199,1108,3377
0.9375,0.1458,1119,1215,1099,3415
0.9792,0.1458,1176,1279,1138,3575
0.0208,0.1042,1394,1639,1439,4462
0.0625,0.1042,1308,1526,1337,4163
0.1042,0.1042,1375,1605,1387,4359
0.1458,0.1042,1363,1606,1371,4334
0.1875,0.1042,1308,1568,1329,4202
0.2292,0.1042,1292,1574,1342,4204
0.2708,0.1042,1272,1560,1335,4162
0.3125,0.1042,1241,1549,1327,4112
0.3542,0.1042,1260,1600,1380,4234
0.3958,0.1042,1161,1455,1271,3879
0.4375,0.1042,1188,1482,1311,3975
0.4792,0.1042,1001,1230,1103,3324
0.5208,0.1042,1093,1374,1250,3708
0.5625,0.1042,1140,1439,1317,3888
0.6042,0.1042,1112,1385,1281,3769
0.6458,0.1042,1125,1391,1295,3800
0.6875,0.1042,1111,1352,1254,3707
0.7292,0.1042,1146,1388,1285,3808
0.7708,0.1042,1160,1382,1272,3803
0.8125,0.1042,1178,1377,1265,3809
0.8542,0.1042,1230,1440,1318,3975
0.8958,0.1042,1159,1343,1220,3709
0.9375,0.1042,1171,1361,1225,3746
0.9792,0.1042,1208,1382,1227,3805
0.0208,0.0625,1341,1576,1384,4294
0.0625,0.0625,1289,1507,1328,4115
0.1042,0.0625,1047,1199,1052,3286
0.1458,0.0625,1506,1817,1568,4890
0.1875,0.0625,1354,1635,1411,4395
0.2292,0.0625,1221,1474,1281,3971
0.2708,0.0625,1310,1603,1381,4290
0.3125,0.0625,1242,1539,1340,4114
0.3542,0.0625,1205,1487,1299,3985
0.3958,0.0625,1237,1525,1342,4098
0.4375,0.0625,1176,1454,1287,3910
0.4792,0.0625,1157,1428,1270,3847
0.5208,0.0625,1083,1326,1201,3601
0.5625,0.0625,1055,1290,1169,3503
0.6042,0.0625,1173,1445,1318,3928
0.6458,0.0625,1215,1489,1367,4061
0.6875,0.0625,1257,1536,1410,4194
0.7292,0.0625,1250,1525,1399,4165
0.7708,0.0625,1230,1471,1352,4042
0.8125,0.0625,1279,1519,1389,4176
0.8542,0.0625,1281,1516,1382,4168
0.8958,0.0625,1327,1573,1426,4313
0.9375,0.0625,1258,1479,1322,4049
0.9792,0.0625,1294,1502,1333,4118
0.0208,0.0208,1352,1625,1444,4414
0.0625,0.0208,1334,1597,1415,4340
0.1042,0.0208,968,1119,996,3071
0.1458,0.0208,1483,1870,1652,5018
0.1875,0.0208,1430,1739,1538,4703
0.2292,0.0208,1326,1601,1417,4339
0.2708,0.0208,1384,1700,1500,4575
0.3125,0.0208,1341,1628,1439,4402
0.3542,0.0208,1333,1629,1441,4398
0.3958,0.0208,1165,1430,1266,3862
0.4375,0.0208,1241,1498,1338,4067
0.4792,0.0208,1171,1401,1254,3813
0.5208,0.0208,1143,1376,1229,3734
0.5625,0.0208,1083,1282,1155,3505
0.6042,0.0208,1290,1566,1407,4252
0.6458,0.0208,1193,1428,1287,3887
0.6875,0.0208,1205,1423,1286,3898
0.7292,0.0208,1069,1251,1134,3432
0.7708,0.0208,975,1133,1027,3109
0.8125,0.0208,918,1067,967,2930
0.8542,0.0208,989,1167,1057,3188
0.8958,0.0208,1159,1344,1220,3703
0.9375,0.0208,1244,1468,1319,4010
0.9792,0.0208,1171,1368,1234,3752
"""

In [ ]:
column_names = ["hue", "sat", "R", "G", "B", "K"]

def load_robot_csv(csv_text):
    df = pd.read_csv(io.StringIO(csv_text), header=None, names=column_names)
    # Real hardware logs occasionally have a truncated row (a dropped serial packet, for
    # example). Drop any row missing a value rather than let it silently become NaN and
    # poison every downstream computation.
    before = len(df)
    df = df.dropna()
    dropped = before - len(df)
    if dropped:
        print(f"  dropped {dropped} incomplete row(s)")
    return df

pooled_cal_text = [CELESTE_CAL, TIDAL_CAL, PACIFIC_BLUE_CAL]
pooled_ver_text = [CELESTE_VER, TIDAL_VER, PACIFIC_BLUE_VER]

train_df = pd.concat([load_robot_csv(t) for t in pooled_cal_text], ignore_index=True)
eval_df = pd.concat([load_robot_csv(t) for t in pooled_ver_text], ignore_index=True)

# Held out completely: never touched until the final generalization check.
holdout_robot_df = load_robot_csv(REDWOOD_CAL)

print("Pooled training rows (celeste + tidal + pacific_blue):", len(train_df))
print("Pooled evaluation rows (same 3 robots, held-out session):", len(eval_df))
print("Held-out robot rows (redwood, never trained on):", len(holdout_robot_df))
train_df.head()


## Step 2: Split into inputs (X) and targets (Y)

Our inputs are the 4 raw sensor channels. We will train **two separate shallow networks**:

- `model_hue`: predicts hue (as sin/cos, see Step 4) from [R, G, B, K]
- `model_sat`: predicts saturation from [R, G, B, K]


In [ ]:
feature_cols = ["R", "G", "B", "K"]

X_train_raw = train_df[feature_cols].values.astype("float32")
X_eval_raw = eval_df[feature_cols].values.astype("float32")
X_holdout_raw = holdout_robot_df[feature_cols].values.astype("float32")

y_hue_train = train_df["hue"].values.astype("float32")
y_hue_eval = eval_df["hue"].values.astype("float32")
y_hue_holdout = holdout_robot_df["hue"].values.astype("float32")

y_sat_train = train_df["sat"].values.astype("float32")
y_sat_eval = eval_df["sat"].values.astype("float32")
y_sat_holdout = holdout_robot_df["sat"].values.astype("float32")

print("X_train_raw shape:", X_train_raw.shape)
print("X_eval_raw shape:", X_eval_raw.shape)
print("X_holdout_raw shape:", X_holdout_raw.shape)


## Step 3: Normalize the inputs, with *fixed* scales

The raw R, G, B, K counts can run into the thousands, while hue and saturation are already
between 0 and 1. Neural networks train more reliably when all inputs sit on a similar,
small scale.

The obvious way to do this is to take the min and max of each channel from the training data
and stretch that range onto [0, 1]. **That turns out to be a mistake here, and it is worth
understanding why**, because it is the single biggest thing standing between this notebook and
a good result.

Different robots have different sensor gains. Redwood, the robot we hold out, reads noticeably
brighter than the three we train on:

| channel | celeste training range | redwood range |
|---|---|---|
| R | 141 - 1147 | 147 - 1506 |
| G | 237 - 1397 | 250 - 1870 |
| K | 868 - 3900 | 946 - 5018 |

If we fit the scaling to the training robots' range, every redwood reading above that range
gets clipped to exactly 1.0. A large chunk of redwood's brightest colors collapse onto the same
input value, and the network cannot tell them apart no matter how well it trained.

So instead we use **fixed, hand-chosen offsets and scales**, wide enough to cover any robot's
sensor. These are the same constants used in the lab's MATLAB calibration script: subtract 100
and divide by 1500 for R, G, B; subtract 500 and divide by 4500 for K. They are not fitted to
any dataset, which means a brand-new robot gets the exact same treatment as a training robot.

This also sidesteps the data-leakage question entirely. There is nothing fitted, so there is
nothing to leak.


In [ ]:
# Fixed scales, not fitted to any dataset. Chosen wide enough to cover any robot's sensor.
CHANNEL_OFFSET = np.array([100.0, 100.0, 100.0, 500.0], dtype="float32")
CHANNEL_SCALE = np.array([1500.0, 1500.0, 1500.0, 4500.0], dtype="float32")

def normalize(X):
    return np.clip((X - CHANNEL_OFFSET) / CHANNEL_SCALE, 0.0, 1.0)

X_train = normalize(X_train_raw)
X_eval = normalize(X_eval_raw)
X_holdout = normalize(X_holdout_raw)

print("Normalized training X, first row:", X_train[0])

# Sanity check: how much of each set would a train-fitted scaling have clipped?
train_max = X_train_raw.max(axis=0)
clipped_fixed = np.mean(X_holdout >= 1.0) * 100
clipped_fitted = np.mean(X_holdout_raw > train_max) * 100
print(f"\nRedwood values pinned at the ceiling under fixed scales:  {clipped_fixed:.1f}%")
print(f"Redwood values that a train-fitted scaling would clip:    {clipped_fitted:.1f}%")


## Step 4: Handle hue's wraparound with sin/cos

Hue lives on a circle: a value of 0.99 is right next to 0.01, not far from it, but plain mean
squared error on the raw hue number would tell the network the opposite. The standard fix is
to predict two numbers instead of one:

`hue_sin = sin(2*pi*hue)`, `hue_cos = cos(2*pi*hue)`

Both live in a comfortable, non-wrapping range, and a point near the top of the circle and a
point near the bottom of the circle now sit close together in (sin, cos) space, exactly like
they do on the real color wheel. To turn a (sin, cos) prediction back into a single hue value,
we use `atan2`, which is the standard way to recover an angle from its sine and cosine.


In [ ]:
def hue_to_sincos(hue):
    angle = 2 * np.pi * hue
    return np.sin(angle), np.cos(angle)

def sincos_to_hue(sin_val, cos_val):
    angle = np.arctan2(sin_val, cos_val)
    hue = angle / (2 * np.pi)
    hue[hue < 0] += 1.0  # wrap negative angles back into [0, 1)
    return hue

y_hue_sin_train, y_hue_cos_train = hue_to_sincos(y_hue_train)
y_hue_sin_eval, y_hue_cos_eval = hue_to_sincos(y_hue_eval)
y_hue_sin_holdout, y_hue_cos_holdout = hue_to_sincos(y_hue_holdout)

# Stack sin and cos together as a single (N, 2) target the network can output at once.
Y_hue_train = np.stack([y_hue_sin_train, y_hue_cos_train], axis=1)
Y_hue_eval = np.stack([y_hue_sin_eval, y_hue_cos_eval], axis=1)

print("Example hue 0.0208 ->", hue_to_sincos(np.array([0.0208])))
print("Example hue 0.98    ->", hue_to_sincos(np.array([0.98])))


## Step 5: Augment the training data

Real sensor readings are noisy: reading the exact same color twice will not give the exact
same RGBK numbers. We can make the network more robust to that noise using only the data we
already have, by creating extra training examples with a little random Gaussian noise added
to the RGBK inputs (standard deviation 15 sensor counts, small next to the hundreds-to-
thousands range the channels actually span). Each noisy copy keeps the *same* true
hue/saturation target, since we are not actually changing the color, only simulating sensor
jitter around it.

This is a simple, common trick: it does not require collecting any new data, but it does
require that the noise you add is smaller than the real variation you expect the sensor to
have.


In [ ]:
def augment_with_noise(X_raw, Y, noise_counts=15.0, copies_per_row=3, rng=None):
    """Add noise in RAW sensor-count space, then normalize.

    Note the order: we perturb the raw counts and push them through normalize()
    afterwards, rather than jittering already-normalized values. This way the noise
    means something physical (about 15 sensor counts of jitter) and lands on the
    same footing as a genuine repeat reading would.
    """
    rng = rng or np.random.default_rng(SEED)
    augmented_X = [normalize(X_raw)]
    augmented_Y = [Y]
    for _ in range(copies_per_row):
        noisy_raw = X_raw + rng.normal(0.0, noise_counts, size=X_raw.shape).astype("float32")
        augmented_X.append(normalize(noisy_raw))
        augmented_Y.append(Y)
    return np.concatenate(augmented_X, axis=0), np.concatenate(augmented_Y, axis=0)

X_train_aug, Y_hue_train_aug = augment_with_noise(X_train_raw, Y_hue_train, copies_per_row=3)
_, y_sat_train_aug = augment_with_noise(X_train_raw, y_sat_train.reshape(-1, 1), copies_per_row=3)
y_sat_train_aug = y_sat_train_aug.flatten()

print("Training rows before augmentation:", len(X_train))
print("Training rows after augmentation: ", len(X_train_aug))


## Step 6: A small architecture sweep

Rather than guessing one network shape, we try a short list of candidate hidden-layer sizes
for each target, train each, and keep the best.

Three details that matter more than the candidate list itself:

**We select on a validation split carved out of the training data, not on the eval set.**
If we picked the winner by its eval-set score and then reported that same score as our result,
the number would be optimistically biased: we would have chosen whichever model happened to
suit that particular data. Keras carves the split out for us with `validation_split=0.15`.

**`tanh` activations rather than `relu`.** The hue network's outputs are sines and cosines,
living in [-1, 1] and varying smoothly. `tanh` matches that shape naturally, where `relu`
(flat below zero, unbounded above) has to work against it. This is also what the lab's MATLAB
`feedforwardnet` uses by default.

**Early stopping.** Instead of committing to a fixed epoch count and hoping it was right,
we train with room to spare and stop when validation loss stops improving, restoring the
best weights seen. A too-small fixed budget underfits; a too-large one overfits. Early
stopping removes the guess.

This is a **small** sweep on purpose. A real project might run hundreds of candidates (see
the lab's `comprehensive_search.m`, which does exactly that at much larger scale). The idea
is identical; only the size differs. Watch the printed scores as it runs, and note how close
together the candidates land. That is a real result, and we come back to it in Step 7.

**Why the two candidate lists are different sizes.** The obvious guess is that hue and
saturation are equally hard, so they should get the same network. They are not. Sweeping
seven architectures over the held-out redwood robot gave this (mean holdout error over two
random seeds, so treat gaps under about 0.003 as noise):

| hidden layers | params (hue) | hue holdout | params (sat) | sat holdout |
|---|---|---|---|---|
| [2, 2, 2] | 28 | 0.049 | 25 | 0.132 |
| [4, 2]    | 36 | 0.051 | 33 | 0.099 |
| [5]       | 37 | 0.061 | 31 | 0.123 |
| [8]       | 58 | 0.054 | 49 | 0.130 |
| [4, 4]    | 50 | 0.052 | 45 | 0.133 |
| [8, 6]    | 108 | 0.051 | 101 | 0.065 |
| [16, 12]  | 310 | 0.047 | 297 | 0.067 |

Hue barely cares. A 28-parameter network, small enough to draw on a whiteboard, lands within
0.002 of one with eleven times as many weights. Saturation has a cliff: everything below
[8, 6] is roughly twice as bad, and then it snaps into place and stops improving. Somewhere
around a hundred parameters the network can finally represent the function, and more capacity
after that buys nothing.

There is a second signal worth learning to read. The tiny saturation networks did not just
score worse, they scored *inconsistently*: their spread across random seeds was ten to twenty
times larger than the hue networks'. When the same architecture on the same data gives you
wildly different answers depending on the starting weights, that is usually the network
telling you it does not have enough capacity to solve the problem, and each run is finding a
different mediocre compromise. High seed variance is evidence of an inadequate model, not
just bad luck.

**Batch size.** We use 64 throughout. Batch size matters more than it looks: it also splits
along the easy/hard line. Hue is fine at any batch size, including full batch (all rows in
one gradient step, no minibatching at all), which reached the best hue number in the whole
study once given enough epochs to make up for taking only one update per epoch. Saturation
clearly wanted minibatches: full batch was 0.16 against 0.09 at batch 16. The gradient noise
that minibatching introduces is doing useful work on the harder target. Full batch would be
tempting here since the dataset is small enough to fit in memory at once, but it needs
thousands of epochs to compete, which is a bad trade in a notebook you re-run often.


In [ ]:
# Hue turns out to be an easy target, so the candidates here are deliberately tiny.
hue_architectures = [
    [2, 2, 2],
    [4, 2],
    [8, 6],
]

# Saturation is harder and falls apart below roughly 100 parameters, so it gets more room.
sat_architectures = [
    [8, 6],
    [16, 12],
    [16, 16, 8],
]

MAX_EPOCHS = 400   # an upper bound; early stopping usually halts well before this

def count_params(hidden_sizes, n_inputs, n_outputs):
    n, total = n_inputs, 0
    for size in hidden_sizes:
        total += n * size + size
        n = size
    return total + n * n_outputs + n_outputs

def build_mlp(hidden_sizes, n_inputs, n_outputs, bounded_output):
    model = keras.Sequential([layers.Input(shape=(n_inputs,))])
    for size in hidden_sizes:
        model.add(layers.Dense(size, activation="tanh"))
    # Hue's sin/cos targets live in [-1, 1], so a tanh output layer fits them directly.
    # Saturation is a plain [0, 1] value, so we leave that output unbounded (linear).
    model.add(layers.Dense(n_outputs, activation="tanh" if bounded_output else None))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="mse")
    return model

def sweep(architectures, X_tr, Y_tr, n_outputs, label, bounded_output):
    print(f"--- Sweeping {label} architectures ---")
    best_model, best_score, best_arch = None, np.inf, None
    for arch in architectures:
        model = build_mlp(arch, X_tr.shape[1], n_outputs, bounded_output)
        history = model.fit(
            X_tr, Y_tr,
            epochs=MAX_EPOCHS,
            batch_size=64,
            verbose=0,
            validation_split=0.15,
            callbacks=[keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=50, restore_best_weights=True)],
        )
        val_loss = min(history.history["val_loss"])
        stopped_at = len(history.history["val_loss"])
        print(f"  hidden layers {str(arch):<14} "
              f"{count_params(arch, X_tr.shape[1], n_outputs):>4} params  "
              f"val MSE = {val_loss:.5f}  (stopped at epoch {stopped_at})")
        if val_loss < best_score:
            best_score, best_model, best_arch = val_loss, model, arch
    print(f"Best {label} architecture: {best_arch} (val MSE = {best_score:.5f})\n")
    return best_model, best_arch

model_hue, best_hue_arch = sweep(
    hue_architectures, X_train_aug, Y_hue_train_aug,
    n_outputs=2, label="hue", bounded_output=True,
)

model_sat, best_sat_arch = sweep(
    sat_architectures, X_train_aug, y_sat_train_aug,
    n_outputs=1, label="saturation", bounded_output=False,
)


## Step 7: Evaluate the chosen networks

Two checks, on purpose:

1. **Pooled evaluation set** (held-out sessions of celeste, tidal, pacific_blue): this tells
   us how well the network fits robots it *has* trained on, just not these exact readings.
2. **Redwood (fully held-out robot)**: this tells us whether the network learned something
   about RGBK -> hue/sat in general, or whether it quietly overfit to the specific sensors it
   was trained on. A much larger error here than on the pooled eval set is a sign of the
   latter.

Hue error is computed circularly (the shorter way around the color wheel), since a raw
subtraction would incorrectly penalize a prediction of 0.99 against a true value of 0.01.

**A caution about the sweep.** Look back at how close the candidates in Step 6 scored. Re-run
this notebook with a different `SEED` and the winner will often change, while the final
accuracy barely moves. On this dataset the architecture is simply not the thing that matters:
anything in the rough range we tried works about equally well, and the run-to-run spread from
random initialization is as large as the gap between candidates. The fixed normalization in
Step 3 did far more for the result than any architecture choice does.

That is worth internalizing, because it generalizes. A sweep that reports a winner always
reports a winner, whether or not the difference is real. Before believing one, check that its
margin is bigger than the noise, by re-running with several seeds and seeing whether the same
candidate keeps winning.


In [ ]:
def circular_hue_error(pred_hue, true_hue):
    diff = np.abs(pred_hue - true_hue)
    return np.minimum(diff, 1.0 - diff)

def predict_hue(model, X):
    sincos = model.predict(X, verbose=0)
    return sincos_to_hue(sincos[:, 0], sincos[:, 1])

hue_pred_eval = predict_hue(model_hue, X_eval)
hue_pred_holdout = predict_hue(model_hue, X_holdout)

sat_pred_eval = model_sat.predict(X_eval, verbose=0).flatten()
sat_pred_holdout = model_sat.predict(X_holdout, verbose=0).flatten()

hue_mae_eval = np.mean(circular_hue_error(hue_pred_eval, y_hue_eval))
hue_mae_holdout = np.mean(circular_hue_error(hue_pred_holdout, y_hue_holdout))
sat_mae_eval = np.mean(np.abs(sat_pred_eval - y_sat_eval))
sat_mae_holdout = np.mean(np.abs(sat_pred_holdout - y_sat_holdout))

print("                     pooled eval (seen robots)   redwood (unseen robot)")
print(f"Hue MAE:             {hue_mae_eval:.4f}                        {hue_mae_holdout:.4f}")
print(f"Saturation MAE:      {sat_mae_eval:.4f}                        {sat_mae_holdout:.4f}")


## Step 8: A non-ML baseline, for comparison

Before trusting the neural network, it's worth asking: how good would a plain formula have
done? Python's built-in `colorsys.rgb_to_hsv` converts normalized R, G, B directly to hue and
saturation with no training at all. It ignores the K (clear) channel entirely, and assumes
the sensor's R, G, B behave like an idealized camera, which is exactly the assumption we said
earlier might not hold.

We compute this baseline on the same pooled evaluation rows used above, for a fair
comparison.


In [ ]:
X_eval_rgb = X_eval[:, 0:3]  # columns are [R, G, B, K] after normalize()

raw_hue_pred = np.array([colorsys.rgb_to_hsv(*row)[0] for row in X_eval_rgb])
raw_sat_pred = np.array([colorsys.rgb_to_hsv(*row)[1] for row in X_eval_rgb])

raw_hue_mae = np.mean(circular_hue_error(raw_hue_pred, y_hue_eval))
raw_sat_mae = np.mean(np.abs(raw_sat_pred - y_sat_eval))

print(f"Raw formula hue MAE:        {raw_hue_mae:.4f}   (NN hue MAE:        {hue_mae_eval:.4f})")
print(f"Raw formula saturation MAE: {raw_sat_mae:.4f}   (NN saturation MAE: {sat_mae_eval:.4f})")


## Step 9: Visualize raw-formula vs. true, and NN vs. true

Each point is one pooled-evaluation reading. The dashed diagonal line is "perfect prediction"
(predicted == true). The top row is the plain colorsys formula; the bottom row is our trained
neural network. Points closer to the diagonal are better.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

axes[0, 0].scatter(y_hue_eval, raw_hue_pred, alpha=0.7, color="gray")
axes[0, 0].plot([0, 1], [0, 1], "k--", label="perfect prediction")
axes[0, 0].set_xlabel("true hue")
axes[0, 0].set_ylabel("predicted hue")
axes[0, 0].set_title("Raw colorsys formula: hue")
axes[0, 0].legend()

axes[0, 1].scatter(y_sat_eval, raw_sat_pred, alpha=0.7, color="gray")
axes[0, 1].plot([0, 1], [0, 1], "k--", label="perfect prediction")
axes[0, 1].set_xlabel("true saturation")
axes[0, 1].set_ylabel("predicted saturation")
axes[0, 1].set_title("Raw colorsys formula: saturation")
axes[0, 1].legend()

axes[1, 0].scatter(y_hue_eval, hue_pred_eval, alpha=0.7)
axes[1, 0].plot([0, 1], [0, 1], "k--", label="perfect prediction")
axes[1, 0].set_xlabel("true hue")
axes[1, 0].set_ylabel("predicted hue")
axes[1, 0].set_title("Neural network: hue")
axes[1, 0].legend()

axes[1, 1].scatter(y_sat_eval, sat_pred_eval, alpha=0.7, color="darkorange")
axes[1, 1].plot([0, 1], [0, 1], "k--", label="perfect prediction")
axes[1, 1].set_xlabel("true saturation")
axes[1, 1].set_ylabel("predicted saturation")
axes[1, 1].set_title("Neural network: saturation")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## Step 10: How well does it generalize to a brand-new robot?

The plot below repeats the neural network comparison, but this time on **redwood**, the robot
that was never included in training, augmentation, or the sweep's model selection. Redwood's
sensor reads brighter than any of the three training robots, so this is a genuine test of
whether the network learned something about RGBK to hue/saturation in general.

Expect the points here to be somewhat more scattered than in Step 9's bottom row. That gap is
the honest cost of moving to a sensor the model has never seen, and pooling more robots into
training is the natural way to shrink it. The gap would be considerably worse under a
train-fitted normalization, for the reason laid out in Step 3.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].scatter(y_hue_holdout, hue_pred_holdout, alpha=0.7, color="seagreen")
axes[0].plot([0, 1], [0, 1], "k--", label="perfect prediction")
axes[0].set_xlabel("true hue")
axes[0].set_ylabel("predicted hue")
axes[0].set_title("Neural network on redwood (unseen robot): hue")
axes[0].legend()

axes[1].scatter(y_sat_holdout, sat_pred_holdout, alpha=0.7, color="seagreen")
axes[1].plot([0, 1], [0, 1], "k--", label="perfect prediction")
axes[1].set_xlabel("true saturation")
axes[1].set_ylabel("predicted saturation")
axes[1].set_title("Neural network on redwood (unseen robot): saturation")
axes[1].legend()

plt.tight_layout()
plt.show()


## Things to try next

This notebook is a starting point, not a finished tool. Some natural next steps, roughly in
order of difficulty:

1. **Test the sweep's honesty.** Change `SEED` at the top to a few different values and re-run.
   Does the same architecture keep winning? Does the final accuracy move much? This is the
   check described in Step 7, and it is the most useful thing on this list.
2. **Break the normalization on purpose.** Swap the fixed scales in Step 3 back to
   `X_train_raw.min(axis=0)` / `.max(axis=0)` and watch what happens to the redwood numbers
   specifically, while the pooled eval numbers barely budge. This is the clearest
   demonstration in the notebook of why held-out *robots* test something that held-out *rows*
   do not.
3. **Change the augmentation.** Try `noise_counts=50` or `copies_per_row=10`. Does more
   augmentation help, hurt, or do nothing?
4. **Widen the sweep.** Add more candidates, or sweep learning rate and batch size too.
   Compare against the lab's `comprehensive_search.m` (MATLAB, `feedforwardnet`) to see the
   same idea at full scale.
3. **Add redwood's own verification readings** (not embedded here, only its calibration
   readings are) as a second held-out check, if that data becomes available.
4. **Try a single combined model** with both hue (sin/cos) and saturation as outputs, instead
   of two separate models.
5. **Add K-fold cross-validation** in place of a single fixed eval set, for a more
   trustworthy accuracy estimate.
